# Chapter 2 — Finding the right line (decision boundary)

Continuation of **`logistic-regression-chap1.ipynb`**: same **separable** student roster as Chapter 1 **without** the symmetric noisy points. Same **`EXPORT_FIGSIZE`**, **`EXPORT_DPI`**, **`SAVE_PAD_INCHES`**, and **`fig_to_image` → `save_fig` / `save_gif`** path as Chapter 1. **2D** study–exam figures match chap1’s **`04_dataset_build.gif`** framing: **default** `plt.subplots` margins (**no** `subplots_adjust(**EXPORT_ADJ)`), so axis labels are not clipped; **3D** frames use **`SIG3D_ADJ = EXPORT_ADJ`** like chap1 Scene 8. Same axes and legend styling (**no plot titles** on panels).

Exports land in **`renders/`** under sequential names **`ch2_00_…` through `ch2_36_…`** (execution order: **`ch2_00`–`ch2_16`** story, **`ch2_17`–`ch2_23`** ε baseline + single-panel nudges + **2×3** grid PNGs + **16∶9** build GIFs (**`ch2_23_epsilon_grid_2x3_build_seq_16x9*.gif`**), then **`ch2_24`** (joint **ε** all-parameters **PNG**s + **trim→grow→reveal** GIF ×2 + four smooth line **GIF**s — shift/rotate × numeric vs **w₁,w₂,b** symbolic legend), **`ch2_25`–`ch2_34`** late-story assets, **`ch2_35`–`ch2_36`** Keras gradient descent — same recipe as chap1 **`79_logistic_gradient_descent_plane.gif`**, 2D plane then 3D σ surface). Where parameters move, each asset is saved twice: **`_*_values.*`** = numeric legend, base name = **typographic** (**bold** on the active term + **LaTeX `\\fontsize` only on that term’s weight symbols / coefficients**). Set **`CH2_DISABLE_TEX=1`** for **bold-only** mathtext (no per-term size).

In [ ]:
import io
import os
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import lines as mlines
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3d projection
from PIL import Image, ImageDraw

np.random.seed(7)

OUTPUT_DIR = Path("renders")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Match `logistic-regression-chap1.ipynb` (figure size, DPI, margins, typography) ---
EXPORT_FIGSIZE = (15.0, 9.5)
EXPORT_DPI = 200
EXPORT_ADJ = dict(left=0.04, right=0.96, bottom=0.06, top=0.96)
# Chapter 1 dataset / `fig_to_image` GIFs use **default** `plt.subplots` margins (they do **not** call
# `subplots_adjust(**EXPORT_ADJ)`); that avoids clipping axis labels. Use EXPORT_ADJ only where chap1 does.
SIG3D_ADJ = EXPORT_ADJ  # same name as `logistic-regression-chap1.ipynb` Scene 8 (`SIG3D_ADJ = EXPORT_ADJ`).

FONT_SIZE = 11 * 1.25
AXIS_LABEL_SIZE = 12 * 1.25
LEGEND_SIZE = 20
TITLE_SIZE = 12 * 1.25
NOTE_SIZE = 10 * 1.25
ANNOT_SIZE = 9 * 1.25
SAVE_PAD_INCHES = 0.12

plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelpad": 8.0 * 1.25,
        "axes.titlepad": 10.0 * 1.25,
        "legend.fontsize": LEGEND_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.frameon": True,
        "legend.framealpha": 0.96,
        "legend.borderaxespad": 0.55,
        "legend.labelspacing": 0.35,
        "legend.handlelength": 1.35,
        "legend.handletextpad": 0.65,
        "savefig.pad_inches": SAVE_PAD_INCHES,
    }
)

PASS_COLOR = "#2ca02c"
FAIL_COLOR = "#d62728"
NEUTRAL_COLOR = "#4f4f4f"

CHECK_ICON_PATH = OUTPUT_DIR / "check.png"
CROSS_ICON_PATH = OUTPUT_DIR / "cross.png"


def _ensure_outcome_icons(size=120, line_width=15):
    _sc = size / 96.0
    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.line(
        [(int(round(20 * _sc)), int(round(55 * _sc))), (int(round(42 * _sc)), int(round(76 * _sc))), (int(round(78 * _sc)), int(round(24 * _sc)))],
        fill=(44, 160, 44, 255),
        width=line_width,
    )
    img.save(CHECK_ICON_PATH)

    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.line([(int(round(24 * _sc)), int(round(24 * _sc))), (int(round(72 * _sc)), int(round(72 * _sc)))], fill=(214, 39, 40, 255), width=line_width)
    draw.line([(int(round(72 * _sc)), int(round(24 * _sc))), (int(round(24 * _sc)), int(round(72 * _sc)))], fill=(214, 39, 40, 255), width=line_width)
    img.save(CROSS_ICON_PATH)


_ensure_outcome_icons()
CHECK_ICON = np.asarray(Image.open(CHECK_ICON_PATH).convert("RGBA"))
CROSS_ICON = np.asarray(Image.open(CROSS_ICON_PATH).convert("RGBA"))


def add_outcome_icon(ax, x, y_value, passed, zoom=0.2, alpha=1.0, rotation_deg=0):
    image_array = CHECK_ICON if passed else CROSS_ICON
    if rotation_deg:
        arr = np.asarray(image_array, dtype=np.uint8)
        if int(round(float(rotation_deg) / 180.0)) % 2:
            arr = np.flip(arr, axis=1)
            arr = np.rot90(arr, k=2, axes=(0, 1))
        image_array = arr
    icon = OffsetImage(image_array, zoom=zoom)
    icon.set_alpha(alpha)
    ab = AnnotationBbox(icon, (x, y_value), frameon=False)
    ab.set_clip_on(False)
    ax.add_artist(ab)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))


def fig_to_image(fig, dpi=None, tight_layout=False, transparent=False):
    """Same PNG raster path as `logistic-regression-chap1.ipynb`."""
    if dpi is None:
        dpi = EXPORT_DPI
    buf = io.BytesIO()
    kw = {"format": "png", "dpi": dpi, "pad_inches": SAVE_PAD_INCHES}
    if tight_layout:
        kw["bbox_inches"] = "tight"
    else:
        kw["bbox_inches"] = None
    if transparent:
        kw["transparent"] = True
    fig.savefig(buf, **kw)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGBA" if transparent else "RGB")


def save_gif(images, filename, duration=40):
    if not images:
        raise ValueError("images list is empty")
    rgb = [im.convert("RGB") for im in images]
    w = max(im.width for im in rgb)
    h = max(im.height for im in rgb)
    if any(im.size != (w, h) for im in rgb):
        bg = (255, 255, 255)
        normed = []
        for im in rgb:
            if im.size == (w, h):
                normed.append(im.copy())
            else:
                canvas = Image.new("RGB", (w, h), bg)
                canvas.paste(im, ((w - im.width) // 2, (h - im.height) // 2))
                normed.append(canvas)
        rgb = normed
    rgb[0].save(
        OUTPUT_DIR / filename,
        save_all=True,
        append_images=rgb[1:],
        duration=duration,
        loop=0,
    )


def save_fig(fig, filename, dpi=None, tight_layout=False, transparent=False):
    """Chapter 1 style: raster via `fig_to_image`, then write PNG."""
    im = fig_to_image(fig, dpi=dpi, tight_layout=tight_layout, transparent=transparent)
    im.save(OUTPUT_DIR / filename)


# --- GIF densification: same parameter ranges, finer steps, shorter ms/frame (~higher FPS) ---
GIF_SMOOTH_FACTOR = 4  # ↑ denser samples along the same ranges; paired with ``_gif_dur`` (~↑FPS)


def _gif_dur(base_ms):
    """Per-frame duration when frame count is scaled by ``GIF_SMOOTH_FACTOR``."""
    return max(8, int(round(float(base_ms) / GIF_SMOOTH_FACTOR)))


def _smooth_n(n):
    """Multiply discrete frame/sample counts for smooth parameter sweeps."""
    return max(2, int(round(float(n) * GIF_SMOOTH_FACTOR)))


# --- Separable roster only (no `noisy_symmetric_points` from chap1) ---
separable_points = [
    (2, 3, 0),
    (4, 5, 0), (5, 6, 0),
    (1, 3, 0), (2, 4, 0), (4, 6, 0),
    (1, 4, 0), (3, 6, 0), (1, 6, 0),
    (3, 2, 1),
    (5, 4, 1), (6, 5, 1),
    (4, 2, 1), (6, 4, 1), (3, 1, 1),
    (4, 1, 1), (5, 2, 1), (6, 3, 1),
    (6, 2, 1),
    (6, 1, 1),
]


def unpack_points(point_list):
    arr = np.array(point_list, dtype=float)
    s = arr[:, 0]
    e = arr[:, 1]
    labels = arr[:, 2].astype(int)
    diff = s - e
    return s, e, labels, diff


study_sep, exam_sep, y_sep, z_sep = unpack_points(separable_points)
n_sep = len(study_sep)
xlim = (0, 7)
ylim = (0, 7)
midpoint_shift = (z_sep[y_sep == 0].max() + z_sep[y_sep == 1].min()) / 2.0

_h = 0.02
_gd_st = np.arange(float(xlim[0]), float(xlim[1]) + 1e-9, _h)
_gd_el = np.arange(float(ylim[0]), float(ylim[1]) + 1e-9, _h)
ST, EL = np.meshgrid(_gd_st, _gd_el)

# Coarser grid for **2D** contour GIFs (``contourf``); fast enough for long animations.
_h_anim = 0.07
_ga_st = np.arange(float(xlim[0]), float(xlim[1]) + 1e-9, _h_anim)
_ga_el = np.arange(float(ylim[0]), float(ylim[1]) + 1e-9, _h_anim)
ST_A, EL_A = np.meshgrid(_ga_st, _ga_el)

# Denser mesh **only** for 3D ``plot_surface`` σ skins (same ``xlim`` / ``ylim``; ↑ ``SURF_RES_MULT`` ⇒ finer).
SURF_RES_MULT = 2.5
_h_3d_surf = float(_h_anim) / float(SURF_RES_MULT)
_ga_st_3d = np.arange(float(xlim[0]), float(xlim[1]) + 1e-9, _h_3d_surf)
_ga_el_3d = np.arange(float(ylim[0]), float(ylim[1]) + 1e-9, _h_3d_surf)
ST_3D, EL_3D = np.meshgrid(_ga_st_3d, _ga_el_3d)

cvals = [0, 0.5, 1]
colors = ["red", "white", "green"]
_norm = plt.Normalize(min(cvals), max(cvals))
_tuples = list(zip(map(_norm, cvals), colors))
CMAP_GD = mpl.colors.LinearSegmentedColormap.from_list("gd_sigmoid_plane", _tuples, 256)

EPS = 0.01
W_ST0, W_EL0, B0 = 1.0, -1.0, 0.0

# Use LaTeX for **style** (typographic) legends so we can scale fonts; set ``CH2_DISABLE_TEX=1`` to force mathtext-only (bold, no per-term size).
CH2_LEGEND_TEX = shutil.which("latex") is not None and os.environ.get("CH2_DISABLE_TEX", "") == ""


def logits_plane(w_st, w_el, b, st, el):
    return b + w_st * st + w_el * el


def boundary_line_xy(w_st, w_el, b, xa, xb, ya, yb):
    if abs(w_el) < 1e-9:
        if abs(w_st) < 1e-9:
            return None
        x0 = -b / w_st
        return np.array([x0, x0]), np.array([ya, yb])
    xs = np.linspace(xa, xb, 400)
    ys = -(b + w_st * xs) / w_el
    m = (ys >= ya) & (ys <= yb)
    if not np.any(m):
        return xs, ys
    return xs[m], ys[m]


def draw_dataset(ax, study_vals, exam_vals, labels, mask=None, alpha=0.95, *, xlim_pair=None, ylim_pair=None):
    if mask is None:
        mask = np.ones(len(study_vals), dtype=bool)
    for i in np.where(mask)[0]:
        add_outcome_icon(ax, study_vals[i], exam_vals[i], passed=bool(labels[i]), alpha=alpha)
    ax.set_xlim(*(xlim_pair if xlim_pair is not None else xlim))
    ax.set_ylim(*(ylim_pair if ylim_pair is not None else ylim))
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def add_combined_legend(ax, loc="upper left"):
    line_handles, line_labels = ax.get_legend_handles_labels()
    merged_handles = []
    merged_labels = []
    seen = set()
    for h, lbl in zip(line_handles, line_labels):
        if lbl and lbl not in seen:
            merged_handles.append(h)
            merged_labels.append(lbl)
            seen.add(lbl)
    if merged_handles:
        ax.legend(handles=merged_handles, labels=merged_labels, loc=loc, prop={"size": LEGEND_SIZE})


def add_threshold_line(ax, shift=0.0, label=None, style="--", color=NEUTRAL_COLOR, linewidth=1.0, x_range=None):
    x0, x1 = x_range if x_range is not None else xlim
    x = np.linspace(x0, x1, 200)
    y_line = x - shift
    ax.plot(x, y_line, style, color=color, linewidth=linewidth, label=label)


def shade_pass_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    top = np.minimum(y_line, yb)
    bot = np.full_like(xs, ya)
    ax.fill_between(xs, bot, top, where=(top > bot), alpha=alpha, color=PASS_COLOR, linewidth=0, zorder=0)


def shade_fail_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    bot2 = np.maximum(y_line, ya)
    ax.fill_between(xs, bot2, yb, where=(yb > bot2), alpha=alpha, color=FAIL_COLOR, linewidth=0, zorder=0)


def mistake_mask(study, exam, y, w_st, w_el, b):
    logit = logits_plane(w_st, w_el, b, study, exam)
    pred = (logit >= 0.0).astype(int)
    return pred != y


def highlight_mistakes(ax, study, exam, y, w_st, w_el, b, radius=0.32, lw=2.8):
    wrong = mistake_mask(study, exam, y, w_st, w_el, b)
    for i in np.where(wrong)[0]:
        circ = plt.Circle(
            (study[i], exam[i]),
            radius,
            facecolor="none",
            edgecolor="red",
            linewidth=lw,
            zorder=12,
        )
        ax.add_patch(circ)


def highlight_mask_indices(ax, study, exam, mask_bool, radius=0.32, lw=2.8):
    """Red circles for a fixed boolean mask (e.g. cumulative misclassifications)."""
    for i in np.where(mask_bool)[0]:
        circ = plt.Circle(
            (study[i], exam[i]),
            radius,
            facecolor="none",
            edgecolor="red",
            linewidth=lw,
            zorder=12,
        )
        ax.add_patch(circ)


def draw_probability_panel(
    ax,
    w_st,
    w_el,
    b,
    legend_label,
    *,
    show_contour=True,
    highlight_eps=False,
    highlight_mask=None,
    st_grid=None,
    el_grid=None,
    legend_tex=False,
):
    stg = ST if st_grid is None else st_grid
    elg = EL if el_grid is None else el_grid
    Z = sigmoid(logits_plane(w_st, w_el, b, stg, elg))
    if show_contour:
        ax.contourf(stg, elg, Z, alpha=0.4, cmap=CMAP_GD, vmin=0, vmax=1, zorder=1)
    bxy = boundary_line_xy(w_st, w_el, b, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, c="grey", linestyle="--", linewidth=1.8, label=legend_label, zorder=3)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    if highlight_eps:
        highlight_mistakes(ax, study_sep, exam_sep, y_sep, w_st, w_el, b)
    elif highlight_mask is not None:
        highlight_mask_indices(ax, study_sep, exam_sep, highlight_mask)
    ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
    if legend_tex:
        finalize_style_legend_tex(ax)


def finalize_style_legend_tex(ax):
    """Style legends use ``\\fontsize`` via LaTeX (matplotlib mathtext does not support ``\\fontsize``)."""
    if not CH2_LEGEND_TEX:
        return
    leg = ax.get_legend()
    if leg is None:
        return
    for t in leg.get_texts():
        t.set_usetex(True)


def math_legend_ax(ax, lines, loc="upper left"):
    handles = [mlines.Line2D([], [], linestyle="none", markersize=0, label=t) for t in lines]
    ax.legend(handles=handles, loc=loc, prop={"size": LEGEND_SIZE}, labelspacing=0.9)


# Fixed-size fragments in LaTeX legends use ``LEGEND_SIZE`` so they match **`ch2_01`** (`legend.fontsize`).


def _math_fs_em(mag_ref):
    """Scaled font size for the parameter that is actively changing (maps ``mag_ref`` like gain / |coefficient|)."""
    mag_ref = float(max(float(mag_ref), 1e-6))
    t = float(np.clip(mag_ref / 2.0, 0.0, 1.0))
    return float(np.clip(9.0 + 19.0 * t, 9.0, 30.0))


def _legend_em_fs(mag_ref):
    """Emphasized weight/coefficient size; never below ``LEGEND_SIZE`` (same baseline as **ch2_01**)."""
    return max(_math_fs_em(mag_ref), float(LEGEND_SIZE))


def _tex_sized_math(inner, fs):
    """LaTeX fragment: one math chunk at a fixed font size (inner is math **without** outer dollars)."""
    fs = float(fs)
    fs2 = fs + 2.0
    return rf"\fontsize{{{fs:.2f}}}{{{fs2:.2f}}}\selectfont $" + inner + "$"


def legend_linear_equation_values(ws, we, b, *, difference_form=False):
    """Numeric coefficients (max 2 decimals) for the active boundary line.

    If ``difference_form`` is True, show ``w_1\,ST - w_2\,EL = 0`` with **no bias** and
    ``w_2 = |w_{EL}|`` (sign of ``we`` ignored for the displayed coefficient).
    """
    ws, we, b = float(ws), float(we), float(b)
    if difference_form:
        return rf"${ws:.2f}\,\mathrm{{ST}} - {abs(we):.2f}\,\mathrm{{EL}} = 0$"
    if abs(b) < 1e-9 and abs(ws + we) < 1e-6:
        return rf"${ws:.2f}\,\mathrm{{ST}} - {ws:.2f}\,\mathrm{{EL}} = 0$"
    return rf"${ws:.2f}\,\mathrm{{ST}}{we:+.2f}\,\mathrm{{EL}}{b:+.2f} = 0$"


def legend_linear_equation_style(ws, we, b, emphasize, mag_ref, *, difference_form=False):
    """
    Typographic legend for ``w·(ST, EL) + b``. With ``difference_form``, **no bias**: render
    ``w₁\,ST - w₂\,EL = 0`` (EL coefficient magnitude only in the **values** companion).

    Use ``emphasize="both"`` to scale **w₁** and **w₂** together (e.g. global gain); otherwise
    only the emphasized weight / ``b`` grows with ``mag_ref``. **ST**, **EL**, operators, **= 0** stay at
    ``LEGEND_SIZE``. Without LaTeX: single ``$...$`` mathtext.
    """
    ws, we, b = float(ws), float(we), float(b)
    fs_nm = float(LEGEND_SIZE)
    fs_em = _legend_em_fs(mag_ref)

    if emphasize == "both":
        fs_w1 = fs_w2 = fs_em
    else:
        fs_w1 = fs_em if emphasize == "st" else fs_nm
        fs_w2 = fs_em if emphasize == "el" else fs_nm
    fs_wb = fs_em if emphasize == "b" else fs_nm

    if emphasize == "both":
        w1_m = r"\mathbf{w}_{1}"
        w2_m = r"\mathbf{w}_{2}"
    else:
        w1_m = r"\mathbf{w}_{1}" if emphasize == "st" else r"w_{1}"
        w2_m = r"\mathbf{w}_{2}" if emphasize == "el" else r"w_{2}"
    b_m = r"\mathbf{b}" if emphasize == "b" else r"b"

    if difference_form:
        if CH2_LEGEND_TEX:
            return (
                _tex_sized_math(w1_m, fs_w1)
                + _tex_sized_math(r"\,\mathrm{ST}", fs_nm)
                + " "
                + _tex_sized_math(r"-", fs_nm)
                + " "
                + _tex_sized_math(w2_m, fs_w2)
                + _tex_sized_math(r"\,\mathrm{EL}", fs_nm)
                + " "
                + _tex_sized_math(r"= 0", fs_nm)
            )
        term_st = w1_m + r"\,\mathrm{ST}"
        term_el = w2_m + r"\,\mathrm{EL}"
        return "$" + term_st + r" - " + term_el + r" = 0$"

    if CH2_LEGEND_TEX:
        return (
            _tex_sized_math(w1_m, fs_w1)
            + _tex_sized_math(r"\,\mathrm{ST}", fs_nm)
            + " "
            + _tex_sized_math(r"+", fs_nm)
            + " "
            + _tex_sized_math(w2_m, fs_w2)
            + _tex_sized_math(r"\,\mathrm{EL}", fs_nm)
            + " "
            + _tex_sized_math(r"+", fs_nm)
            + " "
            + _tex_sized_math(b_m, fs_wb)
            + " "
            + _tex_sized_math(r"= 0", fs_nm)
        )
    term_st = w1_m + r"\,\mathrm{ST}"
    term_el = w2_m + r"\,\mathrm{EL}"
    term_b = b_m
    body = term_st + r" + " + term_el + r" + " + term_b + r" = 0"
    return "$" + body + "$"


def epsilon_emphasize_mag(ws, we, bb):
    ds = abs(float(ws) - W_ST0)
    de = abs(float(we) - W_EL0)
    db = abs(float(bb) - B0)
    if ds >= de and ds >= db and ds > 1e-9:
        return "st", ds
    if de >= ds and de >= db and de > 1e-9:
        return "el", de
    return "b", max(db, 1e-6)


def parallel_threshold_label_values(ws, we, bb, g, *, smooth):
    """Threshold legend (logit ``= 0``). When ``smooth`` + LaTeX: only **numeric weights**
    scale with ``g``; **ST/EL**, operators, and ``= 0`` use ``LEGEND_SIZE`` (matches **ch2_01**)."""
    ws, we, bb = float(ws), float(we), float(bb)
    g = float(g)
    if smooth and CH2_LEGEND_TEX:
        fs_nm = float(LEGEND_SIZE)
        fs_em = _legend_em_fs(g)
        if abs(bb) < 1e-9 and abs(ws + we) < 1e-6:
            return (
                _tex_sized_math(rf"{ws:.2f}", fs_em)
                + _tex_sized_math(r"\,\mathrm{ST}", fs_nm)
                + _tex_sized_math(r"-", fs_nm)
                + _tex_sized_math(rf"{ws:.2f}", fs_em)
                + _tex_sized_math(r"\,\mathrm{EL}", fs_nm)
                + " "
                + _tex_sized_math(r"= 0", fs_nm)
            )
        if abs(bb) < 1e-9:
            return (
                _tex_sized_math(rf"{ws:.2f}", fs_em)
                + _tex_sized_math(r"\,\mathrm{ST}", fs_nm)
                + _tex_sized_math(rf"{we:+.2f}", fs_em)
                + _tex_sized_math(r"\,\mathrm{EL}", fs_nm)
                + " "
                + _tex_sized_math(r"= 0", fs_nm)
            )
        return (
            _tex_sized_math(rf"{ws:.2f}", fs_em)
            + _tex_sized_math(r"\,\mathrm{ST}", fs_nm)
            + _tex_sized_math(rf"{we:+.2f}", fs_em)
            + _tex_sized_math(r"\,\mathrm{EL}", fs_nm)
            + _tex_sized_math(rf"{bb:+.2f}", fs_em)
            + " "
            + _tex_sized_math(r"= 0", fs_nm)
        )
    if abs(bb) < 1e-9 and abs(ws + we) < 1e-6:
        inner = rf"{ws:.2f}\,\mathrm{{ST}} - {ws:.2f}\,\mathrm{{EL}} = 0"
    elif abs(bb) < 1e-9:
        inner = rf"{ws:.2f}\,\mathrm{{ST}}{we:+.2f}\,\mathrm{{EL}} = 0"
    else:
        inner = rf"{ws:.2f}\,\mathrm{{ST}}{we:+.2f}\,\mathrm{{EL}}{bb:+.2f} = 0"
    return "$" + inner + "$"


def parallel_threshold_label_style(g, *, smooth):
    """Typographic threshold legend. When ``smooth``, only **w₁ / w₂** scale with ``g``;
    **ST, EL, +, = 0** stay ``LEGEND_SIZE`` (**``ch2_01``** baseline)."""
    fs_nm = float(LEGEND_SIZE)
    fs_w = _legend_em_fs(g) if smooth else fs_nm
    if CH2_LEGEND_TEX:
        return (
            _tex_sized_math(r"\mathbf{w}_{1}", fs_w)
            + _tex_sized_math(r"\,\mathrm{ST}", fs_nm)
            + " "
            + _tex_sized_math(r"+", fs_nm)
            + " "
            + _tex_sized_math(r"\mathbf{w}_{2}", fs_w)
            + _tex_sized_math(r"\,\mathrm{EL}", fs_nm)
            + " "
            + _tex_sized_math(r"= 0", fs_nm)
        )
    st = r"\mathbf{w}_{1}\,\mathrm{ST}"
    el = r"\mathbf{w}_{2}\,\mathrm{EL}"
    body = st + r" + " + el + r" = 0"
    return "$" + body + "$"


print(f"Chapter 2 students (separable only): {n_sep}")
print("Midpoint shift:", midpoint_shift)

## Dataset intro — **`ch2_00`–`ch2_03`**, **`ch2_00c`–`ch2_00e`**, **`ch2_01b`–`ch2_01g`**

- **`ch2_00_dataset_build.gif`**: same cadence as chap1 **`04_dataset_build.gif`** (no legend).
- **`ch2_00_dataset_only.png`**: static roster — **no** threshold line, **no** legend.
- **`ch2_00b_dataset_st_minus_el_labels.gif`**: same **reveal cadence** as **`ch2_00`**, but each visible student is **annotated** with **ST − EL** (**`z_sep`**); dashed **$\mathrm{ST} - \mathrm{EL} = 0$** + combined legend on every frame.
- **`ch2_00c_dataset_st_el_3d_sigmoid_morph.gif`**: **3D** — while points **rise** on **z = ST − EL**, a faint **grey z = 0** plane plus **vertical** floor-to-point guides show height; those drop away once the **tilted logit sheet** appears; then **morph** to **σ** with **z**-limits easing to **[0, 1]**.
- **`ch2_00d_reverse_77_from_00c_to_2d.gif`**: **reverse chap1 `77`** — opens on the **closing pose** of **`ch2_00c`** (**σ** sheet, **elev 32° / azim −152°**), then **orbit** (**−315°** azimuth sweep — **45°** short of a full turn), **tilt** to top-down, **blend** into a shrunk **2D** σ panel, **expand** the axes (inverse of **77** shrink), and hold on the full **2D** view (threshold + legend like **`ch2_01`**).
- **`ch2_00e_after_00d_parallel_st_el_levels_1_to_4.gif`**: same final **2D** σ view as **`ch2_00d`**, then adds parallel lines **ST − EL = 1, 2, 3, 4**; **legend** shows **only** **ST − EL = 0** (threshold).
- **`ch2_01_dataset_st_el_0.png`**: separable set + dashed **$\mathrm{ST} - \mathrm{EL} = 0$**.
- **`ch2_thumbnail_representative_2d.png`**: **`EXPORT_FIGSIZE`** at **480** DPI (**≈3456×2976** px) — **σ(ST−EL)** **`contourf`** with a **brighter**, **high‑opacity** pass/fail colormap (for **dark** backgrounds), dashed threshold **lw 3**, **transparent** PNG, **bold** axis labels **`loc='left'`** / **`loc='bottom'`**, **`labelpad=2`** (**no legend**).
- **`ch2_01b_dataset_st_eq_el.png`**: same framing as **`ch2_01`**; dashed **class-separating** line with legend **ST = EL** (chap1 **`20_dataset_overview.png`** style).
- **`ch2_01c_fail_point_crosses_st_eq_el_and_back.gif`**: like chap1 **`05`** / **`32`**: moving point at fixed exam **3 h** crosses the line **out and back**; dashed threshold legend **$\mathrm{ST} - \mathrm{EL} = 0$** (same as **`ch2_01`**).
- **`ch2_01d_st_minus_el_zero_pass_half_green.png`**: line **$\mathrm{ST} - \mathrm{EL} = 0$**; **pass** side (**$\mathrm{ST} - \mathrm{EL} > 0$**) shaded **green**.
- **`ch2_01e_st_minus_el_zero_fail_half_red.png`**: same line; **fail** side shaded **red**.
- **`ch2_01f_dataset_st_minus_2el_zero.png`**: same **(ST, EL)** points as **`ch2_01`**; dashed **$\mathrm{ST} - 2\mathrm{EL} = 0$**; each student is shown **pass**/**fail** by which side of that line they lie on (**ST − 2EL ≥ 0** ⇒ pass).
- **`ch2_01g_dataset_st_minus_3el_zero.png`**: same with **$\mathrm{ST} - 3\mathrm{EL} = 0$** and labels from **ST − 3EL ≥ 0**.
- **`ch2_02_multiply_both_sides.gif`**, **`ch2_03_multiply_both_sides_contour.gif`**: multiply-both-sides story (line / contour).

In [2]:
# --- ch2_00–ch2_03 config/helpers (each export in its own cell below) ---
rng_build = np.random.default_rng(7)
idx_pass_req = int(np.where((study_sep == 3) & (exam_sep == 2) & (y_sep == 1))[0][0])
idx_fail_req = int(np.where((study_sep == 2) & (exam_sep == 3) & (y_sep == 0))[0][0])
rest = [i for i in range(n_sep) if i not in (idx_pass_req, idx_fail_req)]
rng_build.shuffle(rest)
reveal_order = [idx_pass_req, idx_fail_req] + rest

DATASET_EMPTY_HOLD = 24
DATASET_FIRST_TWO_HOLD = 48
DATASET_REST_HOLD = 2


def _append_build(mask, n_repeat, out):
    for _ in range(n_repeat):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_dataset(ax, study_sep, exam_sep, y_sep, mask=mask)
        out.append(fig_to_image(fig))


def _frames_ch2_00():
    frames = []
    mask = np.zeros(n_sep, dtype=bool)
    _append_build(mask, DATASET_EMPTY_HOLD, frames)
    for k in range(1, n_sep + 1):
        mask = np.zeros(n_sep, dtype=bool)
        mask[reveal_order[:k]] = True
        hold = DATASET_FIRST_TWO_HOLD if k <= 2 else DATASET_REST_HOLD
        _append_build(mask, hold, frames)
    return frames


def _fmt_st_minus_el(z):
    z = float(z)
    if abs(z - round(z)) < 1e-9:
        return str(int(round(z)))
    return f"{z:.1f}"


def _append_build_st_minus_el_labels(mask, n_repeat, out):
    for _ in range(n_repeat):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_dataset(ax, study_sep, exam_sep, y_sep, mask=mask)
        add_threshold_line(
            ax,
            shift=midpoint_shift,
            label=r"$\mathrm{ST} - \mathrm{EL} = 0$",
            color="black",
            linewidth=1.8,
            style="--",
        )
        dx, dy = 0.22, 0.16
        for i in np.where(mask)[0]:
            ax.text(
                float(study_sep[i]) + dx,
                float(exam_sep[i]) + dy,
                _fmt_st_minus_el(z_sep[i]),
                fontsize=NOTE_SIZE - 1,
                color="#1a1a1a",
                ha="left",
                va="bottom",
                zorder=8,
                clip_on=False,
            )
        add_combined_legend(ax, loc="upper left")
        out.append(fig_to_image(fig))


def _frames_ch2_00b_st_minus_el_labels():
    """Same reveal cadence as **ch2_00**; label each visible student with **ST - EL**; dashed threshold + legend."""
    frames = []
    mask = np.zeros(n_sep, dtype=bool)
    _append_build_st_minus_el_labels(mask, DATASET_EMPTY_HOLD, frames)
    for k in range(1, n_sep + 1):
        mask = np.zeros(n_sep, dtype=bool)
        mask[reveal_order[:k]] = True
        hold = DATASET_FIRST_TWO_HOLD if k <= 2 else DATASET_REST_HOLD
        _append_build_st_minus_el_labels(mask, hold, frames)
    return frames


def _fig_ch2_01():
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    xs = np.linspace(xlim[0], xlim[1], 200)
    ax.plot(xs, xs - midpoint_shift, "k--", linewidth=1.8, label=r"$\mathrm{ST} - \mathrm{EL} = 0$")
    add_combined_legend(ax, loc="upper left")
    return fig


def _fig_ch2_01b_st_eq_el_legend():
    """Same geometry as **ch2_01**; threshold legend reads **ST = EL** (chap1 **`20_dataset_overview.png`** style)."""
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1.8, style="--")
    add_combined_legend(ax, loc="upper left")
    return fig


def _frames_ch2_01c_fail_point_crosses_st_eq_el_and_back():
    """Like chap1 **`05_fail_point_crosses_threshold.gif`** / **`32_fail_point_crosses_threshold_st_el_0.gif`**: masked fail slot + moving icon; **there and back**; dashed line + **ST − EL = 0** legend (same math string as **ch2_01**)."""
    EXAM_ANIM = 3.0
    ST_START, ST_END = 2.0, 3.25
    N_ANIM = 55
    PUSH_HOLD = 5
    fwd = np.linspace(ST_START, ST_END, N_ANIM)
    back = np.linspace(ST_END, ST_START, N_ANIM)[1:]
    st_values = np.concatenate([fwd, np.full(PUSH_HOLD, ST_END), back, np.full(PUSH_HOLD, ST_START)])
    static_mask = np.ones(n_sep, dtype=bool)
    static_mask[idx_fail_req] = False
    frames = []
    for st in st_values:
        passed = (st - EXAM_ANIM) > 1e-6
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_dataset(ax, study_sep, exam_sep, y_sep, mask=static_mask, alpha=0.28)
        add_threshold_line(
            ax,
            shift=midpoint_shift,
            label="ST \u2212 EL = 0",
            color="black",
            linewidth=1.8,
            style="--",
        )
        add_outcome_icon(ax, st, EXAM_ANIM, passed=passed, zoom=0.3, alpha=1.0)
        add_combined_legend(ax, loc="upper left")
        frames.append(fig_to_image(fig))
    return frames


def _fig_ch2_01d_st_minus_el_zero_pass_green():
    """True diagonal **ST − EL = 0**; **pass** half-plane (**ST − EL > 0** wedge) shaded **green**."""
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    shade_pass_half(ax, shift=0.0)
    add_threshold_line(ax, shift=0.0, label=r"$\mathrm{ST} - \mathrm{EL} = 0$", color="black", linewidth=1.8, style="--")
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_combined_legend(ax, loc="upper left")
    return fig


def _fig_ch2_01e_st_minus_el_zero_fail_red():
    """Same diagonal; **fail** half-plane shaded **red**."""
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    shade_fail_half(ax, shift=0.0)
    add_threshold_line(ax, shift=0.0, label=r"$\mathrm{ST} - \mathrm{EL} = 0$", color="black", linewidth=1.8, style="--")
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_combined_legend(ax, loc="upper left")
    return fig


def _fig_ch2_01f_dataset_st_minus_2el_zero():
    """Same **(ST, EL)** roster; dashed **ST − 2EL = 0**; icons **pass** iff **ST − 2EL ≥ 0** (side of line)."""
    logit = study_sep - 2.0 * exam_sep
    y_line = (logit >= 0.0).astype(int)
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_line)
    bxy = boundary_line_xy(1.0, -2.0, 0.0, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, "k--", linewidth=1.8, label=r"$\mathrm{ST} - 2\mathrm{EL} = 0$")
    add_combined_legend(ax, loc="upper left")
    return fig


def _fig_ch2_01g_dataset_st_minus_3el_zero():
    """Same roster; dashed **ST − 3EL = 0**; icons **pass** iff **ST − 3EL ≥ 0**."""
    logit = study_sep - 3.0 * exam_sep
    y_line = (logit >= 0.0).astype(int)
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_line)
    bxy = boundary_line_xy(1.0, -3.0, 0.0, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, "k--", linewidth=1.8, label=r"$\mathrm{ST} - 3\mathrm{EL} = 0$")
    add_combined_legend(ax, loc="upper left")
    return fig


w2, el2, bb2 = 1.0, -1.0, 0.0
multiply_steps_plain = [
    (r"$\mathrm{ST} - \mathrm{EL} = 0$",),
    (r"$2(\mathrm{ST} - \mathrm{EL}) = 2 \cdot 0$",),
    (r"$2\mathrm{ST} - 2\mathrm{EL} = 0$",),
]


def frames_multiply_gif(*, with_contour):
    out = []
    holds = (10, 14, 18)
    scale = 1.0
    for step_lines, nh in zip(multiply_steps_plain, holds):
        for _ in range(nh):
            fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
            if with_contour:
                Z = sigmoid(logits_plane(w2*scale, el2*scale, bb2*scale, ST_A, EL_A))
                ax.contourf(ST_A, EL_A, Z, alpha=0.4, cmap=CMAP_GD, vmin=0, vmax=1, zorder=1)
            bxy = boundary_line_xy(w2, el2, bb2, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
            if bxy is not None:
                bx, by = bxy
                ax.plot(bx, by, c="grey", linestyle="--", linewidth=1.8, zorder=3)
            draw_dataset(ax, study_sep, exam_sep, y_sep)
            math_legend_ax(ax, step_lines)
            out.append(fig_to_image(fig))
        scale = 2.0
    return out


In [ ]:
save_gif(_frames_ch2_00(), "ch2_00_dataset_build.gif", duration=35)

In [ ]:
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
save_fig(fig, "ch2_00_dataset_only.png")

In [ ]:
save_gif(_frames_ch2_00b_st_minus_el_labels(), "ch2_00b_dataset_st_minus_el_labels.gif", duration=35)

## Dataset — 3D **ST − EL** → **σ** (`ch2_00c`)

- **`ch2_00c_dataset_st_el_3d_sigmoid_morph.gif`**: 3D scatter — faint **grey z = 0** floor + **vertical** guides from the floor to each rising point until the **logit sheet** appears; then **`CMAP_GD`**-colored **σ** on patches, **morph** to the curved **σ** surface; camera **elev 32°**, **azim −152°**.

In [3]:
# --- ch2_00c — 3D: points rise on z = ST−EL, σ-colored logit plane, morph to σ surface ---
# Prerequisites: Cell 1 + dataset config cell (**`reveal_order`**, **`ST_A`** / **`EL_A`**, **`SIG3D_ADJ`**, …).


def _frames_ch2_00c_dataset_st_el_3d_sigmoid_morph():
    """Scatter on z = ST−EL, then a tilted **logit** sheet (**Z = ST − EL**) painted by **σ(ST−EL)**; morph **Z** and **z-limits** to the curved **σ** surface."""
    Z_log = ST_A - EL_A
    S_m = sigmoid(Z_log)
    z_lo_m = float(Z_log.min())
    z_hi_m = float(Z_log.max())
    z_lo_raw = min(0.0, z_lo_m, float(np.min(z_sep)))
    z_hi_raw = max(0.0, z_hi_m, float(np.max(z_sep)))
    pad = 0.45
    zlim0 = (z_lo_raw - pad, z_hi_raw + pad)

    Zc = 0.25 * (
        Z_log[:-1, :-1] + Z_log[1:, :-1] + Z_log[:-1, 1:] + Z_log[1:, 1:]
    )
    _norm01 = plt.Normalize(0.0, 1.0)
    fc_sigma = CMAP_GD(_norm01(sigmoid(Zc)))

    elev, azim = 32.0, -152.0
    zs_sigma = sigmoid(z_sep.astype(float))

    _z0_plane_n = 28

    def draw_axes(ax, z_lo, z_hi, zlbl):
        ax.view_init(elev=elev, azim=azim)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_zlim(z_lo, z_hi)
        ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
        ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
        ax.set_zlabel(zlbl, fontsize=AXIS_LABEL_SIZE, labelpad=8)
        ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)

    def draw_z0_plane_and_height_lines(ax, zs_arr):
        """Grey **z = 0** reference sheet + vertical segments from the floor to each marker (orthogonal to the floor)."""
        gx = np.linspace(float(xlim[0]), float(xlim[1]), _z0_plane_n)
        gy = np.linspace(float(ylim[0]), float(ylim[1]), _z0_plane_n)
        GX, GY = np.meshgrid(gx, gy)
        GZ = np.zeros_like(GX, dtype=float)
        ax.plot_surface(
            GX,
            GY,
            GZ,
            color=(0.72, 0.72, 0.72),
            alpha=0.14,
            linewidth=0,
            antialiased=True,
            shade=False,
            zorder=1,
        )
        for i in range(n_sep):
            zi = float(zs_arr[i])
            if zi <= 1e-9:
                continue
            st, el = float(study_sep[i]), float(exam_sep[i])
            ax.plot(
                [st, st],
                [el, el],
                [0.0, zi],
                color="#5c5c5c",
                linewidth=1.35,
                alpha=0.42,
                zorder=2,
            )

    def snap(zs_arr, *, Z_surf, show_surface, surf_alpha, z_lo, z_hi, zlbl):
        fig = plt.figure(figsize=EXPORT_FIGSIZE)
        ax = fig.add_subplot(111, projection="3d")
        fig.subplots_adjust(**SIG3D_ADJ)
        if not show_surface:
            draw_z0_plane_and_height_lines(ax, zs_arr)
        if show_surface and Z_surf is not None:
            ax.plot_surface(
                ST_A,
                EL_A,
                Z_surf,
                facecolors=fc_sigma,
                rstride=1,
                cstride=1,
                linewidth=0,
                antialiased=True,
                shade=False,
                alpha=float(surf_alpha),
            )
        for i in range(n_sep):
            c = PASS_COLOR if y_sep[i] else FAIL_COLOR
            ax.scatter(
                [float(study_sep[i])],
                [float(exam_sep[i])],
                [float(zs_arr[i])],
                color=c,
                s=54,
                depthshade=True,
                zorder=6,
            )
        draw_axes(ax, z_lo, z_hi, zlbl)
        return fig_to_image(fig)

    out = []
    zs = np.zeros(n_sep, dtype=float)
    z_pre = r"$\mathrm{ST}-\mathrm{EL}$"
    z_post = r"$\sigma(\mathrm{ST}-\mathrm{EL})$"

    HOLD_ZERO = 20
    RISE_N = 9
    HOLD_RISEN = 14
    HOLD_PLANE = 16
    MORPH_N = 52
    HOLD_END = 20

    for _ in range(HOLD_ZERO):
        out.append(
            snap(
                zs,
                Z_surf=None,
                show_surface=False,
                surf_alpha=0.0,
                z_lo=zlim0[0],
                z_hi=zlim0[1],
                zlbl=z_pre,
            )
        )

    for j in reveal_order:
        tgt = float(z_sep[j])
        for k in range(RISE_N):
            t = float(k + 1) / RISE_N
            zc = zs.copy()
            zc[j] = tgt * t
            out.append(
                snap(
                    zc,
                    Z_surf=None,
                    show_surface=False,
                    surf_alpha=0.0,
                    z_lo=zlim0[0],
                    z_hi=zlim0[1],
                    zlbl=z_pre,
                )
            )
        zs[j] = tgt

    for _ in range(HOLD_RISEN):
        out.append(
            snap(
                zs,
                Z_surf=None,
                show_surface=False,
                surf_alpha=0.0,
                z_lo=zlim0[0],
                z_hi=zlim0[1],
                zlbl=z_pre,
            )
        )

    for _ in range(HOLD_PLANE):
        out.append(
            snap(
                zs,
                Z_surf=Z_log,
                show_surface=True,
                surf_alpha=0.48,
                z_lo=zlim0[0],
                z_hi=zlim0[1],
                zlbl=z_pre,
            )
        )

    for m in range(MORPH_N):
        t = float(m) / max(MORPH_N - 1, 1)
        Zs = (1.0 - t) * Z_log + t * S_m
        z_lo_t = (1.0 - t) * zlim0[0] + t * (-0.02)
        z_hi_t = (1.0 - t) * zlim0[1] + t * 1.02
        zsc = (1.0 - t) * zs + t * zs_sigma
        lbl = z_post if t >= 0.5 else z_pre
        out.append(
            snap(
                zsc,
                Z_surf=Zs,
                show_surface=True,
                surf_alpha=0.48,
                z_lo=z_lo_t,
                z_hi=z_hi_t,
                zlbl=lbl,
            )
        )

    last = out[-1]
    for _ in range(HOLD_END):
        out.append(last.copy())

    return out



In [34]:
save_gif(_frames_ch2_00c_dataset_st_el_3d_sigmoid_morph(), "ch2_00c_dataset_st_el_3d_sigmoid_morph.gif", duration=40)

## Dataset — reverse **77** from **`ch2_00c`** (`ch2_00d`)

- **`ch2_00d_reverse_77_from_00c_to_2d.gif`**: opens on the **closing 3D σ** pose from **`ch2_00c`**, then **reverse‑77** motion into a full **2D** σ panel (see dataset intro bullet).

In [4]:
# --- ch2_00d — reverse of chap1 ``77_from27_topdown_reveal_lift_3d_reverse64.gif`` ---
# Starts where **`ch2_00c`** ends (**σ(ST−EL)** on **`ST_A`**, **elev 32° / azim −152°**), then roughly **reverses** export **77**:
# hold → **orbit** (azimuth sweeps **−315°** — stops **45°** short of a full **−360°**) → **tilt up** to top-down → **blend** to shrunk **2D** σ → **expand** (inverse **77** shrink) → hold full **2D** (like **`ch2_01`**).
#
# Prerequisites: Cell 1 + dataset config (**`ST_A`**, **`SIG3D_ADJ`**, **`midpoint_shift`**, …).

import matplotlib as mpl
from PIL import Image

SIG3D_TOP_ELEV1 = 89.0
_CH2_77REV_END_SCALE_W = 0.45
_CH2_77REV_END_SCALE_H = 0.72


def _sig2d_smoothstep(u):
    u = float(np.clip(u, 0.0, 1.0))
    return u * u * (3.0 - 2.0 * u)


def _ease_top27(t):
    t = np.clip(t, 0.0, 1.0)
    return 0.5 - 0.5 * np.cos(np.pi * t)


def _shrink_bbox_forward(u_lin, w0, h0, cx, cy, end_scale_w, end_scale_h):
    u_lin = float(np.clip(u_lin, 0.0, 1.0))
    p = _sig2d_smoothstep(_ease_top27(u_lin))
    sw = 1.0 + (float(end_scale_w) - 1.0) * p
    sh = 1.0 + (float(end_scale_h) - 1.0) * p
    w = w0 * sw
    h = h0 * sh
    return mpl.transforms.Bbox.from_bounds(cx - 0.5 * w, cy - 0.5 * h, w, h)


def _bb_lerp(bb0, bb1, t):
    t = float(np.clip(t, 0.0, 1.0))
    x0 = bb0.x0 * (1.0 - t) + bb1.x0 * t
    y0 = bb0.y0 * (1.0 - t) + bb1.y0 * t
    w = bb0.width * (1.0 - t) + bb1.width * t
    h = bb0.height * (1.0 - t) + bb1.height * t
    return mpl.transforms.Bbox.from_bounds(x0, y0, w, h)


def _draw_ch2_sigma_topdown_2d(ax):
    Zs = sigmoid(ST_A - EL_A)
    ax.contourf(
        ST_A,
        EL_A,
        Zs,
        levels=np.linspace(0.0, 1.0, 45),
        cmap=CMAP_GD,
        vmin=0,
        vmax=1,
        antialiased=True,
        alpha=0.32,
        zorder=1,
    )
    add_threshold_line(
        ax,
        shift=midpoint_shift,
        label=r"$\mathrm{ST} - \mathrm{EL} = 0$",
        color="black",
        linewidth=1.8,
        style="--",
    )
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_combined_legend(ax, loc="upper left")


def _snap_3d_sigma_ch2_00c_end(elev, azim, *, hide_z=False):
    Zs = sigmoid(ST_A - EL_A)
    zpt = sigmoid(z_sep.astype(float))
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    fig.subplots_adjust(**SIG3D_ADJ)
    ax.plot_surface(
        ST_A,
        EL_A,
        Zs,
        cmap=CMAP_GD,
        vmin=0,
        vmax=1,
        alpha=0.48,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    for i in range(n_sep):
        c = PASS_COLOR if y_sep[i] else FAIL_COLOR
        ax.scatter(
            [float(study_sep[i])],
            [float(exam_sep[i])],
            [float(zpt[i])],
            color=c,
            s=54,
            depthshade=True,
            zorder=6,
        )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(-0.02, 1.02)
    ax.view_init(elev=float(elev), azim=float(azim))
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
    ax.set_zlabel(r"$\sigma(\mathrm{ST}-\mathrm{EL})$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)
    if hide_z or float(elev) >= 87.5:
        ax.set_zticks([])
        ax.set_zticklabels([])
    return fig_to_image(fig)


def _pil_topdown_in_axes(bb):
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_axes((float(bb.x0), float(bb.y0), float(bb.width), float(bb.height)))
    _draw_ch2_sigma_topdown_2d(ax)
    ax.set_aspect("auto")
    return fig_to_image(fig)


def _frames_ch2_00d_reverse_77_from_00c_to_2d():
    _fig_p, _ax_p = plt.subplots(figsize=EXPORT_FIGSIZE)
    _draw_ch2_sigma_topdown_2d(_ax_p)
    _ax_p.set_aspect("auto")
    _fig_p.canvas.draw()
    pos = _ax_p.get_position()
    bb_full = mpl.transforms.Bbox.from_bounds(
        float(pos.x0), float(pos.y0), float(pos.width), float(pos.height)
    )
    w0, h0 = float(pos.width), float(pos.height)
    cx = float(pos.x0 + 0.5 * pos.width)
    cy = float(pos.y0 + 0.5 * pos.height)
    plt.close(_fig_p)

    bb_shrunk = _shrink_bbox_forward(
        1.0, w0, h0, cx, cy, _CH2_77REV_END_SCALE_W, _CH2_77REV_END_SCALE_H
    )

    out = []
    elev0, az0 = 32.0, -152.0

    HOLD0 = 14
    N_SPIN = 90
    _orbit_az_span_deg = 360.0 - 17.0 - 45.0  # stop orbit 45° before a full revolution
    N_TILT = 48
    HOLD_TOP = 10
    N_BLEND = 26
    N_EXPAND = 84
    HOLD1 = 16

    for _ in range(HOLD0):
        out.append(_snap_3d_sigma_ch2_00c_end(elev0, az0, hide_z=False))

    az_end = float(az0)
    for j in range(1, N_SPIN + 1):
        u = float(j) / float(N_SPIN)
        az = az0 - _orbit_az_span_deg * u
        out.append(_snap_3d_sigma_ch2_00c_end(elev0, az, hide_z=False))
        az_end = az

    for j in range(N_TILT):
        u = _sig2d_smoothstep(j / max(N_TILT - 1, 1))
        elev = elev0 + (SIG3D_TOP_ELEV1 - elev0) * u
        hide = float(elev) >= 87.5
        out.append(_snap_3d_sigma_ch2_00c_end(elev, az_end, hide_z=hide))

    for _ in range(HOLD_TOP):
        out.append(_snap_3d_sigma_ch2_00c_end(SIG3D_TOP_ELEV1, az_end, hide_z=True))

    im3d = out[-1].copy()
    im2d_shrunk = _pil_topdown_in_axes(bb_shrunk)
    nb = max(N_BLEND, 2)
    for k in range(nb):
        u = _sig2d_smoothstep(k / float(nb - 1))
        out.append(Image.blend(im3d, im2d_shrunk, u))

    for j in range(N_EXPAND):
        u = _sig2d_smoothstep(j / max(N_EXPAND - 1, 1))
        bb = _bb_lerp(bb_shrunk, bb_full, u)
        out.append(_pil_topdown_in_axes(bb))

    last = out[-1]
    for _ in range(HOLD1):
        out.append(last.copy())

    return out


def _frames_ch2_00e_after_00d_parallel_st_el_levels_1_to_4():
    """Same closing **2D** σ view as **ch2_00d**, then add parallel **ST − EL = k** for **k = 1…4**; legend is **only** the dashed **ST − EL = 0** threshold."""
    N_HOLD0 = 14
    N_HOLD_PER = 10
    N_HOLD1 = 18
    xs = np.linspace(float(xlim[0]), float(xlim[1]), 400)
    lw_x = 1.35
    col_x = "#666666"

    def _snap(n_extra):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        Zs = sigmoid(ST_A - EL_A)
        ax.contourf(
            ST_A,
            EL_A,
            Zs,
            levels=np.linspace(0.0, 1.0, 45),
            cmap=CMAP_GD,
            vmin=0,
            vmax=1,
            antialiased=True,
            alpha=0.32,
            zorder=1,
        )
        ys0 = xs - float(midpoint_shift)
        (h0,) = ax.plot(xs, ys0, "k--", linewidth=1.8, label="ST \u2212 EL = 0", zorder=4)
        for kk in range(1, int(n_extra) + 1):
            ys = xs - float(kk)
            m = (ys >= float(ylim[0])) & (ys <= float(ylim[1]))
            ax.plot(xs[m], ys[m], "--", color=col_x, linewidth=lw_x, zorder=2)
        draw_dataset(ax, study_sep, exam_sep, y_sep)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.legend(handles=[h0], loc="upper left", prop={"size": LEGEND_SIZE})
        return fig_to_image(fig)

    out = []
    for _ in range(N_HOLD0):
        out.append(_snap(0))
    for nt in range(1, 5):
        for _ in range(N_HOLD_PER):
            out.append(_snap(nt))
    for _ in range(N_HOLD1):
        out.append(_snap(4))
    return out


In [62]:
save_gif(_frames_ch2_00d_reverse_77_from_00c_to_2d(), "ch2_00d_reverse_77_from_00c_to_2d.gif", duration=38)


## Dataset — after **`ch2_00d`** (`ch2_00e`)

- **`ch2_00e_after_00d_parallel_st_el_levels_1_to_4.gif`**: opens on the same **2D** σ + students pose as the **end** of **`ch2_00d`**, then draws parallel dashed lines **ST − EL = 1, 2, 3, 4** in order; **legend = threshold only** (**ST − EL = 0**).

In [64]:
save_gif(
    _frames_ch2_00e_after_00d_parallel_st_el_levels_1_to_4(),
    "ch2_00e_after_00d_parallel_st_el_levels_1_to_4.gif",
    duration=38,
)


In [ ]:
save_fig(_fig_ch2_01(), "ch2_01_dataset_st_el_0.png")

In [5]:
# Representative **2D thumbnail** — **`EXPORT_FIGSIZE`**, **σ** field + threshold (**lw 3**), transparent PNG at **480** DPI (**brighter**, near‑opaque **`contourf`** for dark page backgrounds).
def _fig_ch2_thumbnail_representative_2d():
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    fig.patch.set_facecolor("none")
    fig.patch.set_alpha(0.0)
    ax.set_facecolor("none")
    Z = sigmoid(logits_plane(W_ST0, W_EL0, B0, ST_A, EL_A))
    # Saturated pass/fail hues + bright mid‑tone so **σ** reads on **dark** compositing backgrounds.
    _thumb_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "ch2_thumb_sig",
        ["#ff3355", "#fff6a8", "#2aff7a"],
        256,
    )
    ax.contourf(
        ST_A,
        EL_A,
        Z,
        levels=np.linspace(0.0, 1.0, 56),
        cmap=_thumb_cmap,
        vmin=0,
        vmax=1,
        antialiased=True,
        alpha=0.94,
        zorder=1,
    )
    add_threshold_line(
        ax,
        shift=midpoint_shift,
        label=None,
        color="black",
        linewidth=3.0,
        style="--",
    )
    draw_dataset(ax, study_sep, exam_sep, y_sep, alpha=1.0)
    ax.set_xlabel(
        "Study time (hours)",
        fontsize=AXIS_LABEL_SIZE,
        fontweight="bold",
        labelpad=2,
        loc="left",
    )
    ax.set_ylabel(
        "Exam length (hours)",
        fontsize=AXIS_LABEL_SIZE,
        fontweight="bold",
        labelpad=2,
        loc="bottom",
    )
    ax.set_aspect("equal", adjustable="box")
    return fig


save_fig(
    _fig_ch2_thumbnail_representative_2d(),
    "ch2_thumbnail_representative_2d.png",
    dpi=480,
    transparent=True,
)



In [ ]:
save_fig(_fig_ch2_01b_st_eq_el_legend(), "ch2_01b_dataset_st_eq_el.png")

In [56]:
save_gif(_frames_ch2_01c_fail_point_crosses_st_eq_el_and_back(), "ch2_01c_fail_point_crosses_st_eq_el_and_back.gif", duration=42)

In [ ]:
save_fig(_fig_ch2_01d_st_minus_el_zero_pass_green(), "ch2_01d_st_minus_el_zero_pass_half_green.png")

In [ ]:
save_fig(_fig_ch2_01e_st_minus_el_zero_fail_red(), "ch2_01e_st_minus_el_zero_fail_half_red.png")

In [69]:
save_fig(_fig_ch2_01f_dataset_st_minus_2el_zero(), "ch2_01f_dataset_st_minus_2el_zero.png")

In [70]:
save_fig(_fig_ch2_01g_dataset_st_minus_3el_zero(), "ch2_01g_dataset_st_minus_3el_zero.png")

In [ ]:
save_gif(frames_multiply_gif(with_contour=False), "ch2_02_multiply_both_sides.gif", duration=38)

In [ ]:
save_gif(frames_multiply_gif(with_contour=True), "ch2_03_multiply_both_sides_contour.gif", duration=38)

## 3D σ scaling — rotate / shift in 2D and 3D

- **`ch2_04_sigmoid_param_scale_3d.gif`**: σ-plane steepness as **k·(1, −1, 0)**; **camera azimuth −152°**.
- **`ch2_04b_sigmoid_scale_2d_smooth_values.gif`** / **`ch2_04b_sigmoid_scale_2d_smooth.gif`**: same **k** path as **`ch2_04`**, **2D** contour; legend **without bias**: **w₁ ST − w₂ EL = 0** (values use **|w_EL|** for the EL coefficient). **`_values`** = numeric coefficients; base name = **bold + fontsize on both w₁ and w₂ together** (gain **k**).
- **`ch2_05_rotate_line_contour_values.gif`** / **`ch2_05_rotate_line_contour.gif`**: **w₁** then **w₂** phased (**1→0.01→2→1**, then **−1→1→−1**); same **no-bias** legend **w₁ ST − w₂ EL = 0**; fontsize tracks **|w₁|** or **|w₂|** on the active segment. **`_values`** = numeric legend; base = typographic emphasis. **Red circles** = misclassified points **each frame**.
- **`ch2_06_shift_bias_contour_values.gif`** / **`ch2_06_shift_bias_contour.gif`**: **b: 0 → −2 → 2 → 0**; same **values / style** split as **`ch2_05`**.
- **`ch2_07_rotate_line_3d.gif`**, **`ch2_08_shift_bias_3d.gif`**: same motions in 3D; **azimuth −152°**.
- **`ch2_09`–`ch2_16`**: parallel level sets **logit = L** for **L ∈ {0,1,2,3}**, gain **g**; **legend = threshold only** (the **L = 0** / decision line). **`ch2_11` / `ch2_12` / `ch2_15` / `ch2_16`** (smooth **g**): **\fontsize** tracks **g** only on **numeric coefficients** (values) or **both weight symbols** (style); **`+`** and **`= 0`** stay base size. **`_*_values.gif`**: numeric threshold; base **`.gif`**: typographic. **`ch2_13`–`ch2_16`** add **σ(logit)** as **contour only** (not in the legend).

In [6]:
SIG3D_MS = _gif_dur(34)


def _style_3d(ax, *, elev, azim, zlabel=r"$\sigma(\mathrm{logit})$"):
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
    ax.set_zlabel(zlabel, fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)


def frame_3d_scaled(k_scale, *, elev=32.0, azim=-152.0):
    w_st, w_el, b = k_scale * W_ST0, k_scale * W_EL0, k_scale * B0
    Zs = sigmoid(logits_plane(w_st, w_el, b, ST_3D, EL_3D))
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    fig.subplots_adjust(**SIG3D_ADJ)
    ax.plot_surface(ST_3D, EL_3D, Zs, cmap=CMAP_GD, vmin=0, vmax=1, alpha=0.42, linewidth=0, antialiased=True, shade=False)
    zs = sigmoid(logits_plane(w_st, w_el, b, study_sep, exam_sep))
    for i in range(n_sep):
        c = PASS_COLOR if y_sep[i] else FAIL_COLOR
        ax.scatter([study_sep[i]], [exam_sep[i]], [zs[i]], color=c, s=55, depthshade=True, zorder=6)
    bxy = boundary_line_xy(w_st, w_el, b, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        z0 = np.full_like(bx, 0.5)
        ax.plot(bx, by, z0, "k--", linewidth=1.4, zorder=4)
    _style_3d(ax, elev=elev, azim=azim)
    return fig_to_image(fig)


k_up = np.linspace(0.35, 2.4, _smooth_n(28))
k_down = np.linspace(2.4, 0.35, _smooth_n(28))
k_hold = np.full(_smooth_n(10), 0.35)
k_seq = np.concatenate([k_up, k_down, k_hold])


def _build_ch2_04b_frames(*, legend_mode):
    frames = []
    for k in k_seq:
        kk = float(k)
        w_st, w_el, b = kk * W_ST0, kk * W_EL0, kk * B0
        if legend_mode == "values":
            leg = legend_linear_equation_values(w_st, w_el, b, difference_form=True)
            lg_tex = False
        else:
            leg = legend_linear_equation_style(w_st, w_el, b, "both", mag_ref=kk, difference_form=True)
            lg_tex = True
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_probability_panel(ax, w_st, w_el, b, leg, show_contour=True, st_grid=ST_A, el_grid=EL_A, legend_tex=lg_tex)
        frames.append(fig_to_image(fig))
    return frames


nph = _smooth_n(22)
seg_st_1 = np.linspace(1.0, 0.01, nph)
seg_st_2 = np.linspace(0.01, 2.0, nph)
seg_st_3 = np.linspace(2.0, 1.0, max(nph // 2, _smooth_n(10) // GIF_SMOOTH_FACTOR))
seg_el_1 = np.linspace(-1.0, 1.0, nph)
seg_el_2 = np.linspace(1.0, -1.0, nph)


def _frame_ws_we_bb(ws, we, bb, legend_label, *, legend_tex=False):
    wrong_now = mistake_mask(study_sep, exam_sep, y_sep, float(ws), float(we), float(bb))
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(
        ax,
        float(ws),
        float(we),
        float(bb),
        legend_label,
        show_contour=True,
        st_grid=ST_A,
        el_grid=EL_A,
        highlight_mask=wrong_now,
        legend_tex=legend_tex,
    )
    return fig_to_image(fig)


def _build_ch2_05_frames(*, legend_mode):
    frames = []
    for ws in np.concatenate([seg_st_1, seg_st_2, seg_st_3]):
        we, bb = W_EL0, B0
        if legend_mode == "values":
            leg = legend_linear_equation_values(ws, we, bb, difference_form=True)
            frames.append(_frame_ws_we_bb(ws, we, bb, leg))
        else:
            leg = legend_linear_equation_style(ws, we, bb, "st", mag_ref=abs(float(ws)), difference_form=True)
            frames.append(_frame_ws_we_bb(ws, we, bb, leg, legend_tex=True))
    for we in np.concatenate([seg_el_1, seg_el_2]):
        ws, bb = W_ST0, B0
        if legend_mode == "values":
            leg = legend_linear_equation_values(ws, we, bb, difference_form=True)
            frames.append(_frame_ws_we_bb(ws, we, bb, leg))
        else:
            leg = legend_linear_equation_style(ws, we, bb, "el", mag_ref=abs(float(we)), difference_form=True)
            frames.append(_frame_ws_we_bb(ws, we, bb, leg, legend_tex=True))
    return frames


n_seg = _smooth_n(32)
b_path = np.concatenate([np.linspace(0.0, -2.0, n_seg), np.linspace(-2.0, 2.0, n_seg), np.linspace(2.0, 0.0, n_seg)])


def _build_ch2_06_frames(*, legend_mode):
    frames = []
    for bb in b_path:
        bb = float(bb)
        wrong_now = mistake_mask(study_sep, exam_sep, y_sep, W_ST0, W_EL0, bb)
        if legend_mode == "values":
            leg = legend_linear_equation_values(W_ST0, W_EL0, bb)
            lg_tex = False
        else:
            leg = legend_linear_equation_style(W_ST0, W_EL0, bb, "b", mag_ref=abs(bb))
            lg_tex = True
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_probability_panel(ax, W_ST0, W_EL0, bb, leg, show_contour=True, st_grid=ST_A, el_grid=EL_A, highlight_mask=wrong_now, legend_tex=lg_tex)
        frames.append(fig_to_image(fig))
    return frames


alpha0 = np.pi / 4.0
d_alpha = 0.65
alphas = np.concatenate([np.linspace(alpha0 - d_alpha, alpha0 + d_alpha, _smooth_n(36)), np.linspace(alpha0 + d_alpha, alpha0 - d_alpha, _smooth_n(36))])


def w_from_alpha(a):
    return float(np.cos(a)), float(-np.sin(a))


def frame_rotate_3d(a, *, elev=36.0, azim=-152.0):
    ws, we = w_from_alpha(a)
    Zs = sigmoid(logits_plane(ws, we, 0.0, ST_3D, EL_3D))
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    fig.subplots_adjust(**SIG3D_ADJ)
    ax.plot_surface(ST_3D, EL_3D, Zs, cmap=CMAP_GD, vmin=0, vmax=1, alpha=0.4, linewidth=0, antialiased=True, shade=False)
    zs = sigmoid(logits_plane(ws, we, 0.0, study_sep, exam_sep))
    for i in range(n_sep):
        c = PASS_COLOR if y_sep[i] else FAIL_COLOR
        ax.scatter([study_sep[i]], [exam_sep[i]], [zs[i]], color=c, s=52, zorder=6)
    bxy = boundary_line_xy(ws, we, 0.0, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, np.full_like(bx, 0.5), "k--", linewidth=1.2, zorder=4)
    _style_3d(ax, elev=elev, azim=azim)
    return fig_to_image(fig)


def frame_shift_3d(bb, *, elev=36.0, azim=-152.0):
    Zs = sigmoid(logits_plane(W_ST0, W_EL0, float(bb), ST_3D, EL_3D))
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    fig.subplots_adjust(**SIG3D_ADJ)
    ax.plot_surface(ST_3D, EL_3D, Zs, cmap=CMAP_GD, vmin=0, vmax=1, alpha=0.4, linewidth=0, antialiased=True, shade=False)
    zs = sigmoid(logits_plane(W_ST0, W_EL0, float(bb), study_sep, exam_sep))
    for i in range(n_sep):
        c = PASS_COLOR if y_sep[i] else FAIL_COLOR
        ax.scatter([study_sep[i]], [exam_sep[i]], [zs[i]], color=c, s=52, zorder=6)
    bxy = boundary_line_xy(W_ST0, W_EL0, float(bb), float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, np.full_like(bx, 0.5), "k--", linewidth=1.2, zorder=4)
    _style_3d(ax, elev=elev, azim=azim)
    return fig_to_image(fig)


def _parallel_lines_frame(g, *, with_contour, note=None, legend_mode="values", smooth=False):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    g = float(g)
    ws, we, bb = g * W_ST0, g * W_EL0, 0.0
    if with_contour:
        Z = sigmoid(logits_plane(ws, we, bb, ST_A, EL_A))
        ax.contourf(ST_A, EL_A, Z, alpha=0.4, cmap=CMAP_GD, vmin=0, vmax=1, zorder=1)
    xs = np.linspace(xlim[0], xlim[1], 400)
    for j, L in enumerate((0, 1, 2, 3)):
        shift = (float(L) / g) if abs(g) > 1e-9 else float(L)
        ys = xs - shift
        m = (ys >= ylim[0]) & (ys <= ylim[1])
        if L == 0:
            lbl = parallel_threshold_label_style(g, smooth=smooth) if legend_mode == "style" else parallel_threshold_label_values(ws, we, bb, g, smooth=smooth)
        else:
            lbl = "_nolegend_"
        ax.plot(xs[m], ys[m], "--", color=PARALLEL_COLORS[j], linewidth=1.75, label=lbl, zorder=3)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    if note:
        ax.text(0.02, 0.98, note, transform=ax.transAxes, va="top", fontsize=NOTE_SIZE)
    add_combined_legend(ax, loc="upper left")
    if legend_mode == "style" or (smooth and CH2_LEGEND_TEX and legend_mode == "values"):
        finalize_style_legend_tex(ax)
    return fig_to_image(fig)


PARALLEL_COLORS = ["black", "#1f77b4", "#2ca02c", "#bcbd22"]
_ph = _smooth_n(6)
_g_up = np.linspace(1.0, 2.0, _smooth_n(40))
_g_dn = np.linspace(2.0, 1.0, _smooth_n(40))


def _parallel_intro_frames(*, with_contour, legend_mode):
    return [_parallel_lines_frame(1.0, with_contour=with_contour, legend_mode=legend_mode) for _ in range(_ph)]


def _parallel_step_frames(*, with_contour, legend_mode):
    return ([_parallel_lines_frame(1.0, with_contour=with_contour, legend_mode=legend_mode) for _ in range(_smooth_n(6))]
            + [_parallel_lines_frame(2.0, with_contour=with_contour, legend_mode=legend_mode) for _ in range(_smooth_n(8))])


def _parallel_smooth_frames(gs, *, with_contour, legend_mode):
    return [_parallel_lines_frame(float(g), with_contour=with_contour, legend_mode=legend_mode, smooth=True) for g in gs]


In [ ]:
save_gif([frame_3d_scaled(float(k)) for k in k_seq], "ch2_04_sigmoid_param_scale_3d.gif", duration=SIG3D_MS)

In [ ]:
save_gif(_build_ch2_04b_frames(legend_mode="values"), "ch2_04b_sigmoid_scale_2d_smooth_values.gif", duration=SIG3D_MS)

In [ ]:
save_gif(_build_ch2_04b_frames(legend_mode="style"), "ch2_04b_sigmoid_scale_2d_smooth.gif", duration=SIG3D_MS)

In [ ]:
save_gif(_build_ch2_05_frames(legend_mode="values"), "ch2_05_rotate_line_contour_values.gif", duration=_gif_dur(28))

In [ ]:
save_gif(_build_ch2_05_frames(legend_mode="style"), "ch2_05_rotate_line_contour.gif", duration=_gif_dur(28))

In [ ]:
save_gif(_build_ch2_06_frames(legend_mode="values"), "ch2_06_shift_bias_contour_values.gif", duration=_gif_dur(26))

In [ ]:
save_gif(_build_ch2_06_frames(legend_mode="style"), "ch2_06_shift_bias_contour.gif", duration=_gif_dur(26))

In [ ]:
save_gif([frame_rotate_3d(float(a)) for a in alphas], "ch2_07_rotate_line_3d.gif", duration=_gif_dur(32))

In [ ]:
save_gif([frame_shift_3d(float(bb)) for bb in b_path], "ch2_08_shift_bias_3d.gif", duration=_gif_dur(28))

In [ ]:
save_gif(_parallel_intro_frames(with_contour=False, legend_mode="values"), "ch2_09_parallel_st_el_lines_intro_values.gif", duration=_gif_dur(36))

In [ ]:
save_gif(_parallel_intro_frames(with_contour=False, legend_mode="style"), "ch2_09_parallel_st_el_lines_intro.gif", duration=_gif_dur(36))

In [ ]:
save_gif(_parallel_step_frames(with_contour=False, legend_mode="values"), "ch2_10_parallel_amplify_step_values.gif", duration=_gif_dur(38))

In [ ]:
save_gif(_parallel_step_frames(with_contour=False, legend_mode="style"), "ch2_10_parallel_amplify_step.gif", duration=_gif_dur(38))

In [ ]:
save_gif(_parallel_smooth_frames(_g_up, with_contour=False, legend_mode="values"), "ch2_11_parallel_amplify_smooth_values.gif", duration=_gif_dur(30))

In [ ]:
save_gif(_parallel_smooth_frames(_g_up, with_contour=False, legend_mode="style"), "ch2_11_parallel_amplify_smooth.gif", duration=_gif_dur(30))

In [ ]:
save_gif(_parallel_smooth_frames(_g_dn, with_contour=False, legend_mode="values"), "ch2_12_parallel_scale_down_smooth_values.gif", duration=_gif_dur(30))

In [ ]:
save_gif(_parallel_smooth_frames(_g_dn, with_contour=False, legend_mode="style"), "ch2_12_parallel_scale_down_smooth.gif", duration=_gif_dur(30))

In [ ]:
save_gif(_parallel_intro_frames(with_contour=True, legend_mode="values"), "ch2_13_parallel_st_el_lines_intro_contour_values.gif", duration=_gif_dur(36))

In [ ]:
save_gif(_parallel_intro_frames(with_contour=True, legend_mode="style"), "ch2_13_parallel_st_el_lines_intro_contour.gif", duration=_gif_dur(36))

In [ ]:
save_gif(_parallel_step_frames(with_contour=True, legend_mode="values"), "ch2_14_parallel_amplify_step_contour_values.gif", duration=_gif_dur(38))

In [ ]:
save_gif(_parallel_step_frames(with_contour=True, legend_mode="style"), "ch2_14_parallel_amplify_step_contour.gif", duration=_gif_dur(38))

In [5]:
save_gif(_parallel_smooth_frames(_g_up, with_contour=True, legend_mode="values"), "ch2_15_parallel_amplify_smooth_contour_values.gif", duration=_gif_dur(30))

In [6]:
save_gif(_parallel_smooth_frames(_g_up, with_contour=True, legend_mode="style"), "ch2_15_parallel_amplify_smooth_contour.gif", duration=_gif_dur(30))

In [7]:
save_gif(_parallel_smooth_frames(_g_dn, with_contour=True, legend_mode="values"), "ch2_16_parallel_scale_down_smooth_contour_values.gif", duration=_gif_dur(30))

In [8]:
save_gif(_parallel_smooth_frames(_g_dn, with_contour=True, legend_mode="style"), "ch2_16_parallel_scale_down_smooth_contour.gif", duration=_gif_dur(30))

## ε nudges — **`ch2_17`–`ch2_24`** (baseline **ST − (5/7)EL − 1 = 0**, **`ch2_` prefix**)

- **`ch2_17_baseline_st_minus_5_7_el_minus_1_values.png`** / **`.png`**: opening line **$\mathrm{ST} - \frac{5}{7}\,\mathrm{EL} - 1 = 0$** (weights **(1, -5/7, -1)**); **values** = numeric legend, **style** = typographic emphasis on **both** weights.
- **`ch2_17b_bce_grad_eps_joint.png`**: single panel — joint step **$(w_{1}+\epsilon\,\partial L/\partial w_{1})\,\mathrm{ST}+\cdots$** at the **`ch2_17`** baseline weights (**mean** binary-crossentropy **$L$** over the roster).
- **`ch2_18`–`ch2_23`**: six stems (**$w_1\pm\epsilon$**, **$w_2\pm\epsilon$**, **$b\pm\epsilon$**) from that baseline; each has **`{stem}_values.png`** (numeric coefficients) and **`{stem}.png`** (symbolic legend **$(w_1\pm\epsilon)\,\mathrm{ST}+\cdots$** / **$(w_2\pm\epsilon)$** / **$(b\pm\epsilon)$**).
- **`ch2_23_epsilon_grid_2x3_values.png`** / **`ch2_23_epsilon_grid_2x3.png`**: **2×3** grid of the six ε nudge panels (**values** vs **symbolic ε** legends).
- **`ch2_23_epsilon_grid_2x3_build_seq_16x9_values.gif`** / **`ch2_23_epsilon_grid_2x3_build_seq_16x9.gif`**: full canvas **16∶9** (letterboxed around the **2×3** block); **each** panel is **`EXPORT_FIGSIZE`** inches with **tight** inter-panel gaps; panels **appear one at a time** (**values** vs **symbolic ε**).
- **`ch2_24_epsilon_combined_all_values.png`** / **`ch2_24_epsilon_combined_all.png`**: among **8** sign patterns **(±ε on each of w₁, w₂, b)** from the baseline, use the one with the **fewest** mistakes on the dataset (**values** = numeric coefficients; base = symbolic **$(w_{1}\pm\epsilon)\,\mathrm{ST}+\cdots$** for that pattern).
- **`ch2_24_epsilon_grid_trim_grow_reveal_combined_values.gif`** / **`ch2_24_epsilon_grid_trim_grow_reveal_combined.gif`**: **first frame = last frame** of **`ch2_23_epsilon_grid_2x3_build_seq_16x9*.gif`** (full **16∶9** **2×3** grid); then **drop** the three nudges **not** used by the combined winner; **grow** the three survivors to **full canvas** (semi-transparent overlap); **crossfade** to the combined panel (**values** vs symbolic ε legends).
- **`ch2_24_epsilon_line_shift_b_smooth.gif`**: smooth **translation** of the line (**$b$** sweeps, **$w_1,w_2$** fixed at the **5/7** baseline); **numeric** legend.
- **`ch2_24_epsilon_line_rotate_w1_w2_smooth.gif`**: smooth **rotation** of **$(w_1,w_2)$** in weight space (**$b$** fixed); **numeric** legend.
- **`ch2_24_epsilon_line_shift_b_smooth_w_notation.gif`** / **`ch2_24_epsilon_line_rotate_w1_w2_smooth_w_notation.gif`**: same motions with **symbolic** threshold legend **$w_{1}\,\mathrm{ST} + w_{2}\,\mathrm{EL} + b = 0$** (**`legend_linear_equation_style`** + **`legend_tex`**).

In [ ]:
# Baseline boundary **ST - (5/7) EL - 1 = 0**  <=>  **w·(ST,EL)+b** with **(w1,w2,b)=(1,-5/7,-1)**.
_BASE57 = (0.1, -1.0, 3.0)
_WST57, _WEL57, _B57 = _BASE57
EPS = .2


def _eps_emphasize_mag_57(ws, we, bb):
    ds = abs(float(ws) - _WST57)
    de = abs(float(we) - _WEL57)
    db = abs(float(bb) - _B57)
    if ds >= de and ds >= db and ds > 1e-9:
        return "st", ds
    if de >= ds and de >= db and de > 1e-9:
        return "el", de
    return "b", max(db, 1e-6)


baseline_stem = "ch2_17_baseline_st_minus_5_7_el_minus_1"
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_probability_panel(
    ax,
    _WST57,
    _WEL57,
    _B57,
    legend_linear_equation_values(_WST57, _WEL57, _B57),
    show_contour=True,
    highlight_eps=True,
)
save_fig(fig, f"{baseline_stem}_values.png")
_mag_baseline = float(max(np.hypot(_WST57, abs(_WEL57)), 1e-6))
_leg_base_style = legend_linear_equation_style(_WST57, _WEL57, _B57, "both", mag_ref=_mag_baseline)
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_probability_panel(
    ax, _WST57, _WEL57, _B57, _leg_base_style, show_contour=True, highlight_eps=True, legend_tex=True
)
save_fig(fig, f"{baseline_stem}.png")


def _bce_grad_mean_binary(w_st, w_el, b, study, exam, y):
    """Mean binary-crossentropy gradients (σ − y) averaged over the roster: ∂L/∂w₁, ∂L/∂w₂, ∂L/∂b."""
    logit = logits_plane(w_st, w_el, b, study, exam)
    sig = sigmoid(logit)
    err = sig - y
    g_st = float(np.mean(err * study))
    g_el = float(np.mean(err * exam))
    g_b = float(np.mean(err))
    return g_st, g_el, g_b


_g_st57, _g_el57, _g_b57 = _bce_grad_mean_binary(_WST57, _WEL57, _B57, study_sep, exam_sep, y_sep)

_w_st_grad = _WST57 + EPS * _g_st57
_w_el_grad = _WEL57 + EPS * _g_el57
_b_grad = _B57 + EPS * _g_b57
_grad_joint_stem = "ch2_17b_bce_grad_eps_joint"
_leg_grad_joint = (
    r"$(w_{1}+\epsilon\,\partial L/\partial w_{1})\,\mathrm{ST}"
    r" + (w_{2}+\epsilon\,\partial L/\partial w_{2})\,\mathrm{EL}"
    r" + (b+\epsilon\,\partial L/\partial b) = 0$"
)
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_probability_panel(
    ax,
    _w_st_grad,
    _w_el_grad,
    _b_grad,
    _leg_grad_joint,
    show_contour=True,
    highlight_eps=True,
)
save_fig(fig, f"{_grad_joint_stem}.png")

epsilon_variants = [
    ("ch2_18_epsilon_w_st_minus", _WST57 - EPS, _WEL57, _B57, r"$(w_{1}-\epsilon)\,\mathrm{ST} + w_{2}\,\mathrm{EL} + b = 0$"),
    ("ch2_19_epsilon_w_st_plus", _WST57 + EPS, _WEL57, _B57, r"$(w_{1}+\epsilon)\,\mathrm{ST} + w_{2}\,\mathrm{EL} + b = 0$"),
    ("ch2_20_epsilon_w_el_minus", _WST57, _WEL57 - EPS, _B57, r"$w_{1}\,\mathrm{ST} + (w_{2}-\epsilon)\,\mathrm{EL} + b = 0$"),
    ("ch2_21_epsilon_w_el_plus", _WST57, _WEL57 + EPS, _B57, r"$w_{1}\,\mathrm{ST} + (w_{2}+\epsilon)\,\mathrm{EL} + b = 0$"),
    ("ch2_22_epsilon_b_minus", _WST57, _WEL57, _B57 - EPS, r"$w_{1}\,\mathrm{ST} + w_{2}\,\mathrm{EL} + (b-\epsilon) = 0$"),
    ("ch2_23_epsilon_b_plus", _WST57, _WEL57, _B57 + EPS, r"$w_{1}\,\mathrm{ST} + w_{2}\,\mathrm{EL} + (b+\epsilon) = 0$"),
]

for stem, ws, we, bb, leg_v in epsilon_variants:
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(
        ax, ws, we, bb, legend_linear_equation_values(ws, we, bb), show_contour=True, highlight_eps=True
    )
    save_fig(fig, f"{stem}_values.png")
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(ax, ws, we, bb, leg_v, show_contour=True, highlight_eps=True)
    save_fig(fig, f"{stem}.png")

# --- Large 2 rows × 3 columns: row 0 = minus ε, row 1 = plus ε; columns w₁, w₂, b ---
GRID_FIGSIZE = (EXPORT_FIGSIZE[0] * 3.0, EXPORT_FIGSIZE[1] * 2.0)
epsilon_grid_pairs = [
    (epsilon_variants[0], epsilon_variants[1]),
    (epsilon_variants[2], epsilon_variants[3]),
    (epsilon_variants[4], epsilon_variants[5]),
]


def _epsilon_grid_figure(*, use_values):
    fig, axs = plt.subplots(2, 3, figsize=GRID_FIGSIZE)
    for col, (minus_spec, plus_spec) in enumerate(epsilon_grid_pairs):
        for row, spec in enumerate((minus_spec, plus_spec)):
            stem, ws, we, bb, leg_v = spec
            if use_values:
                leg = legend_linear_equation_values(ws, we, bb)
            else:
                leg = leg_v
            ax = axs[row, col]
            draw_probability_panel(
                ax,
                ws,
                we,
                bb,
                leg,
                show_contour=True,
                highlight_eps=True,
                legend_tex=False,
            )
    fig.subplots_adjust(left=0.07, right=0.98, top=0.95, bottom=0.12, wspace=0.16, hspace=0.28)
    return fig


save_fig(_epsilon_grid_figure(use_values=True), "ch2_23_epsilon_grid_2x3_values.png")
save_fig(_epsilon_grid_figure(use_values=False), "ch2_23_epsilon_grid_2x3.png")

# Joint ε on **w₁, w₂, b**: among **8** sign patterns **(±ε, ±ε, ±ε)** on the baseline, pick the one
# with the **fewest** mistakes (tie-break: lower **(s₁, s₂, s₃)** in lex order with **−1 < +1**).


def _mistake_n_eps57(ws, we, bb):
    return int(np.sum(mistake_mask(study_sep, exam_sep, y_sep, float(ws), float(we), float(bb))))


def _leg_joint_three_eps(s1, s2, s3):
    o1, o2, o3 = (("+" if s > 0 else "-") for s in (s1, s2, s3))
    return (
        rf"$(w_{{1}}{o1}\epsilon)\,\mathrm{{ST}} + (w_{{2}}{o2}\epsilon)\,\mathrm{{EL}} + (b{o3}\epsilon) = 0$"
    )


_best_key = None
_best_ws = _best_we = _best_bb = None
_best_s123 = (0, 0, 0)
for s1 in (-1, 1):
    for s2 in (-1, 1):
        for s3 in (-1, 1):
            wsv = _WST57 + float(s1) * EPS
            wev = _WEL57 + float(s2) * EPS
            bbv = _B57 + float(s3) * EPS
            n_wrong = _mistake_n_eps57(wsv, wev, bbv)
            key = (n_wrong, s1, s2, s3)
            if _best_key is None or key < _best_key:
                _best_key = key
                _best_ws, _best_we, _best_bb = wsv, wev, bbv
                _best_s123 = (s1, s2, s3)

_w_st_comb, _w_el_comb, _b_comb = _best_ws, _best_we, _best_bb
_leg_comb_sym = _leg_joint_three_eps(*_best_s123)
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_probability_panel(
    ax,
    _w_st_comb,
    _w_el_comb,
    _b_comb,
    legend_linear_equation_values(_w_st_comb, _w_el_comb, _b_comb),
    show_contour=True,
    highlight_eps=True,
)
save_fig(fig, "ch2_24_epsilon_combined_all_values.png")
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_probability_panel(
    ax,
    _w_st_comb,
    _w_el_comb,
    _b_comb,
    _leg_comb_sym,
    show_contour=True,
    highlight_eps=True,
)
save_fig(fig, "ch2_24_epsilon_combined_all.png")

# --- Animation: full **2×3** grid → drop the three ε nudges not used by the combined winner → grow the
# three remaining panels to full canvas (overlapping, semi-transparent) → fade to the combined panel.


def _epsilon_keep_indices(s1, s2, s3):
    """Which of ``epsilon_variants[0..5]`` matches ``(s₁,s₂,s₃)`` for **w₁, w₂, b** (minus vs plus)."""
    return (
        0 if s1 < 0 else 1,
        2 if s2 < 0 else 3,
        4 if s3 < 0 else 5,
    )


def _spec_legend_eps_anim(spec, use_values):
    stem, ws, we, bb, leg_v = spec
    return legend_linear_equation_values(ws, we, bb) if use_values else leg_v


def _epsilon_169_cell_geometry():
    """16∶9 letterboxed **2×3** layout (same as ``_epsilon_grid_figure_partial``). Returns ``FW, FH, rects``."""
    cell_w, cell_h = float(EXPORT_FIGSIZE[0]), float(EXPORT_FIGSIZE[1])
    gap_x, gap_y = 0.05, 0.045
    content_w = 3.0 * cell_w + 2.0 * gap_x
    content_h = 2.0 * cell_h + gap_y
    ar_c = content_w / content_h
    ar_169 = 16.0 / 9.0
    if ar_c >= ar_169:
        FW = content_w
        FH = FW / ar_169
    else:
        FH = content_h
        FW = FH * ar_169
    left0 = 0.5 * (FW - content_w)
    bottom0 = 0.5 * (FH - content_h)
    order_rc = [(0, 0), (1, 0), (0, 1), (1, 1), (0, 2), (1, 2)]
    rects = []
    for r, c in order_rc:
        left_m = left0 + float(c) * (cell_w + gap_x)
        bottom_m = bottom0 + float(1 - r) * (cell_h + gap_y)
        rects.append((left_m / FW, bottom_m / FH, cell_w / FW, cell_h / FH))
    return FW, FH, rects


def _grow_overlap_frame(t, s123, use_values):
    FW, FH, rects = _epsilon_169_cell_geometry()
    keep = _epsilon_keep_indices(*s123)
    fig = plt.figure(figsize=(FW, FH))
    fig.patch.set_facecolor("white")
    for k, idx in enumerate(keep):
        x0, y0, w0, h0 = rects[idx]
        xn = (1.0 - t) * x0
        yn = (1.0 - t) * y0
        wn = (1.0 - t) * w0 + t * 1.0
        hn = (1.0 - t) * h0 + t * 1.0
        ax = fig.add_axes([xn, yn, wn, hn], zorder=10 + k)
        spec = epsilon_variants[idx]
        ws, we, bb = spec[1], spec[2], spec[3]
        leg = _spec_legend_eps_anim(spec, use_values)
        draw_probability_panel(ax, ws, we, bb, leg, show_contour=True, highlight_eps=True, legend_tex=False)
        if t > 0.82:
            ax.patch.set_alpha(0.48 if k == 0 else (0.52 if k == 1 else 1.0))
    return fig_to_image(fig)


def _overlap_three_full_bleed(s123, use_values, wh):
    W, H = wh
    keep = _epsilon_keep_indices(*s123)
    imgs = []
    for idx in keep:
        spec = epsilon_variants[idx]
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        ws, we, bb = spec[1], spec[2], spec[3]
        leg = _spec_legend_eps_anim(spec, use_values)
        draw_probability_panel(ax, ws, we, bb, leg, show_contour=True, highlight_eps=True, legend_tex=False)
        imgs.append(fig_to_image(fig).resize((W, H), Image.Resampling.LANCZOS))
    out = Image.new("RGBA", (W, H), (255, 255, 255, 255))
    for k, im in enumerate(imgs):
        irgba = im.convert("RGBA")
        if k < 2:
            r, g, b, a = irgba.split()
            a = a.point(lambda p, kk=k: int(round(float(p) * (0.44 if kk == 0 else 0.48))))
            irgba = Image.merge("RGBA", (r, g, b, a))
        out = Image.alpha_composite(out, irgba)
    return out.convert("RGB")


def _combined_letterboxed_to_wh(use_values, wh):
    W, H = wh
    leg = (
        legend_linear_equation_values(_w_st_comb, _w_el_comb, _b_comb)
        if use_values
        else _leg_comb_sym
    )
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(
        ax,
        _w_st_comb,
        _w_el_comb,
        _b_comb,
        leg,
        show_contour=True,
        highlight_eps=True,
        legend_tex=False,
    )
    im = fig_to_image(fig)
    iw, ih = im.size
    scale = min(float(W) / float(iw), float(H) / float(ih))
    nw = max(1, int(round(iw * scale)))
    nh = max(1, int(round(ih * scale)))
    im2 = im.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (W, H), (255, 255, 255))
    canvas.paste(im2, ((W - nw) // 2, (H - nh) // 2))
    return canvas


def _frames_ch2_24_eps_grid_trim_grow_reveal_combined(*, use_values):
    hold_intro = max(5, _smooth_n(4))
    hold_trim = max(8, _smooth_n(6))
    hold_overlap = max(8, _smooth_n(7))
    hold_end = max(10, _smooth_n(8))

    out = []

    im_full = _epsilon_grid_figure_partial(6, use_values=use_values)
    W, H = im_full.size
    for _ in range(hold_intro):
        out.append(im_full.copy())

    im_trim = _epsilon_trimmed_grid_figure(_best_s123, use_values)
    for _ in range(hold_trim):
        out.append(im_trim.copy())

    for t in np.linspace(0.0, 1.0, _smooth_n(28)):
        out.append(_grow_overlap_frame(float(t), _best_s123, use_values))

    im_stack = _overlap_three_full_bleed(_best_s123, use_values, (W, H))
    for _ in range(hold_overlap):
        out.append(im_stack.copy())

    im_combined = _combined_letterboxed_to_wh(use_values, (W, H))
    last = out[-1].convert("RGB")
    for u in np.linspace(0.0, 1.0, _smooth_n(20)):
        out.append(Image.blend(last, im_combined, float(u)))

    for _ in range(hold_end):
        out.append(im_combined.copy())

    return out


def _epsilon_grid_figure_partial(n_filled, *, use_values, show_indices=None):
    """First ``n_filled`` panels (build order 1…6), **16∶9** canvas (same as ``ch2_23_epsilon_grid_2x3_build_seq_16x9``).

    If ``show_indices`` is set (subset of ``{0,…,5}``), draw **only** those panels — same canvas, unused slots stay blank.
    """
    FW, FH, _ = _epsilon_169_cell_geometry()
    cell_w, cell_h = float(EXPORT_FIGSIZE[0]), float(EXPORT_FIGSIZE[1])
    gap_x, gap_y = 0.05, 0.045
    content_w = 3.0 * cell_w + 2.0 * gap_x
    content_h = 2.0 * cell_h + gap_y
    left0 = 0.5 * (FW - content_w)
    bottom0 = 0.5 * (FH - content_h)
    fig = plt.figure(figsize=(FW, FH))
    order = [
        (0, 0, epsilon_variants[0]),
        (1, 0, epsilon_variants[1]),
        (0, 1, epsilon_variants[2]),
        (1, 1, epsilon_variants[3]),
        (0, 2, epsilon_variants[4]),
        (1, 2, epsilon_variants[5]),
    ]
    for idx, (r, c, spec) in enumerate(order):
        if show_indices is not None:
            if idx not in show_indices:
                continue
        elif idx >= int(n_filled):
            break
        left_m = left0 + float(c) * (cell_w + gap_x)
        bottom_m = bottom0 + float(1 - r) * (cell_h + gap_y)
        ax = fig.add_axes([left_m / FW, bottom_m / FH, cell_w / FW, cell_h / FH])
        _, ws, we, bb, leg_v = spec
        if use_values:
            leg = legend_linear_equation_values(ws, we, bb)
        else:
            leg = leg_v
        draw_probability_panel(
            ax,
            ws,
            we,
            bb,
            leg,
            show_contour=True,
            highlight_eps=True,
            legend_tex=False,
        )
    return fig_to_image(fig)


def _epsilon_trimmed_grid_figure(s123, use_values):
    """Three surviving ε panels on the **16∶9** grid (unused slots blank)."""
    return _epsilon_grid_figure_partial(
        6, use_values=use_values, show_indices=set(_epsilon_keep_indices(*s123))
    )


save_gif(
    _frames_ch2_24_eps_grid_trim_grow_reveal_combined(use_values=True),
    "ch2_24_epsilon_grid_trim_grow_reveal_combined_values.gif",
    duration=_gif_dur(38),
)
save_gif(
    _frames_ch2_24_eps_grid_trim_grow_reveal_combined(use_values=False),
    "ch2_24_epsilon_grid_trim_grow_reveal_combined.gif",
    duration=_gif_dur(38),
)


def _frames_ch2_23_epsilon_grid_2x3_build_seq_16x9(*, use_values):
    hold_each = max(6, _smooth_n(5))
    hold_last = max(14, _smooth_n(10))
    out = []
    for n in range(1, 7):
        im = _epsilon_grid_figure_partial(n, use_values=use_values)
        nh = hold_each if n < 6 else hold_last
        for _ in range(nh):
            out.append(im.copy())
    return out


save_gif(
    _frames_ch2_23_epsilon_grid_2x3_build_seq_16x9(use_values=True),
    "ch2_23_epsilon_grid_2x3_build_seq_16x9_values.gif",
    duration=_gif_dur(42),
)
save_gif(
    _frames_ch2_23_epsilon_grid_2x3_build_seq_16x9(use_values=False),
    "ch2_23_epsilon_grid_2x3_build_seq_16x9.gif",
    duration=_gif_dur(42),
)


def _eps57_panel_img(ws, we, bb, legend_label, *, legend_tex=False):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(
        ax, ws, we, bb, legend_label, show_contour=True, highlight_eps=True, legend_tex=legend_tex
    )
    return fig_to_image(fig)


def _frames_ch2_24_eps57_line_shift_b_smooth():
    ws0, we0, bb0 = _BASE57
    span = 0.5
    bs = np.concatenate(
        [
            np.full(_smooth_n(6), bb0 - span),
            np.linspace(bb0 - span, bb0 + span, _smooth_n(44)),
            np.full(_smooth_n(6), bb0 + span),
        ]
    )
    out = []
    for bbv in bs:
        bv = float(bbv)
        leg = legend_linear_equation_values(ws0, we0, bv)
        out.append(_eps57_panel_img(ws0, we0, bv, leg, legend_tex=False))
    return out


def _frames_ch2_24_eps57_line_rotate_w1_w2_smooth():
    ws0, we0, bb0 = _BASE57
    w0 = np.array([ws0, we0], dtype=float)
    phis = np.concatenate(
        [
            np.full(_smooth_n(5), -0.4),
            np.linspace(-0.4, 0.4, _smooth_n(44)),
            np.full(_smooth_n(5), 0.4),
        ]
    )
    out = []
    for phi in phis:
        c, s = float(np.cos(phi)), float(np.sin(phi))
        R = np.array([[c, -s], [s, c]], dtype=float)
        w = R @ w0
        wsv, wev = float(w[0]), float(w[1])
        leg = legend_linear_equation_values(wsv, wev, bb0)
        out.append(_eps57_panel_img(wsv, wev, bb0, leg, legend_tex=False))
    return out


def _eps57_panel_img_symbolic(ws, we, bb, emphasize, mag_ref):
    leg_s = legend_linear_equation_style(
        ws, we, bb, emphasize, mag_ref=max(float(mag_ref), 1e-6)
    )
    return _eps57_panel_img(ws, we, bb, leg_s, legend_tex=True)


def _frames_ch2_24_eps57_line_shift_b_smooth_symbolic():
    ws0, we0, bb0 = _BASE57
    span = 0.5
    bs = np.concatenate(
        [
            np.full(_smooth_n(6), bb0 - span),
            np.linspace(bb0 - span, bb0 + span, _smooth_n(44)),
            np.full(_smooth_n(6), bb0 + span),
        ]
    )
    out = []
    for bbv in bs:
        bv = float(bbv)
        mag = max(abs(bv - bb0), float(EPS) * 0.2)
        out.append(_eps57_panel_img_symbolic(ws0, we0, bv, "b", mag))
    return out


def _frames_ch2_24_eps57_line_rotate_w1_w2_smooth_symbolic():
    ws0, we0, bb0 = _BASE57
    w0 = np.array([float(ws0), float(we0)], dtype=float)
    phis = np.concatenate(
        [
            np.full(_smooth_n(5), -0.4),
            np.linspace(-0.4, 0.4, _smooth_n(44)),
            np.full(_smooth_n(5), 0.4),
        ]
    )
    out = []
    for phi in phis:
        c, s = float(np.cos(phi)), float(np.sin(phi))
        R = np.array([[c, -s], [s, c]], dtype=float)
        w = R @ w0
        wsv, wev = float(w[0]), float(w[1])
        mag = max(float(np.hypot(wsv, abs(wev))), 0.35)
        out.append(_eps57_panel_img_symbolic(wsv, wev, bb0, "both", mag))
    return out


save_gif(_frames_ch2_24_eps57_line_shift_b_smooth(), "ch2_24_epsilon_line_shift_b_smooth.gif", duration=_gif_dur(36))
save_gif(_frames_ch2_24_eps57_line_rotate_w1_w2_smooth(), "ch2_24_epsilon_line_rotate_w1_w2_smooth.gif", duration=_gif_dur(34))
save_gif(
    _frames_ch2_24_eps57_line_shift_b_smooth_symbolic(),
    "ch2_24_epsilon_line_shift_b_smooth_w_notation.gif",
    duration=_gif_dur(36),
)
save_gif(
    _frames_ch2_24_eps57_line_rotate_w1_w2_smooth_symbolic(),
    "ch2_24_epsilon_line_rotate_w1_w2_smooth_w_notation.gif",
    duration=_gif_dur(34),
)


In [ ]:
def _epsilon_keep_indices(s1, s2, s3):
    """Which of ``epsilon_variants[0..5]`` matches ``(s₁,s₂,s₃)`` for **w₁, w₂, b** (minus vs plus)."""
    return (
        0 if s1 < 0 else 1,
        2 if s2 < 0 else 3,
        4 if s3 < 0 else 5,
    )


def _spec_legend_eps_anim(spec, use_values):
    stem, ws, we, bb, leg_v = spec
    return legend_linear_equation_values(ws, we, bb) if use_values else leg_v


def _epsilon_169_cell_geometry():
    """16∶9 letterboxed **2×3** layout (same as ``_epsilon_grid_figure_partial``). Returns ``FW, FH, rects``."""
    cell_w, cell_h = float(EXPORT_FIGSIZE[0]), float(EXPORT_FIGSIZE[1])
    gap_x, gap_y = 0.05, 0.045
    content_w = 3.0 * cell_w + 2.0 * gap_x
    content_h = 2.0 * cell_h + gap_y
    ar_c = content_w / content_h
    ar_169 = 16.0 / 9.0
    if ar_c >= ar_169:
        FW = content_w
        FH = FW / ar_169
    else:
        FH = content_h
        FW = FH * ar_169
    left0 = 0.5 * (FW - content_w)
    bottom0 = 0.5 * (FH - content_h)
    order_rc = [(0, 0), (1, 0), (0, 1), (1, 1), (0, 2), (1, 2)]
    rects = []
    for r, c in order_rc:
        left_m = left0 + float(c) * (cell_w + gap_x)
        bottom_m = bottom0 + float(1 - r) * (cell_h + gap_y)
        rects.append((left_m / FW, bottom_m / FH, cell_w / FW, cell_h / FH))
    return FW, FH, rects


def _grow_overlap_frame(t, s123, use_values):
    FW, FH, rects = _epsilon_169_cell_geometry()
    keep = _epsilon_keep_indices(*s123)
    fig = plt.figure(figsize=(FW, FH))
    fig.patch.set_facecolor("white")
    for k, idx in enumerate(keep):
        x0, y0, w0, h0 = rects[idx]
        xn = (1.0 - t) * x0
        yn = (1.0 - t) * y0
        wn = (1.0 - t) * w0 + t * 1.0
        hn = (1.0 - t) * h0 + t * 1.0
        ax = fig.add_axes([xn, yn, wn, hn], zorder=10 + k)
        spec = epsilon_variants[idx]
        ws, we, bb = spec[1], spec[2], spec[3]
        leg = _spec_legend_eps_anim(spec, use_values)
        draw_probability_panel(ax, ws, we, bb, leg, show_contour=True, highlight_eps=True, legend_tex=False)
        if t > 0.82:
            ax.patch.set_alpha(0.48 if k == 0 else (0.52 if k == 1 else 1.0))
    return fig_to_image(fig)


def _overlap_three_full_bleed(s123, use_values, wh):
    W, H = wh
    keep = _epsilon_keep_indices(*s123)
    imgs = []
    for idx in keep:
        spec = epsilon_variants[idx]
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        ws, we, bb = spec[1], spec[2], spec[3]
        leg = _spec_legend_eps_anim(spec, use_values)
        draw_probability_panel(ax, ws, we, bb, leg, show_contour=True, highlight_eps=True, legend_tex=False)
        imgs.append(fig_to_image(fig).resize((W, H), Image.Resampling.LANCZOS))
    out = Image.new("RGBA", (W, H), (255, 255, 255, 255))
    for k, im in enumerate(imgs):
        irgba = im.convert("RGBA")
        if k < 2:
            r, g, b, a = irgba.split()
            a = a.point(lambda p, kk=k: int(round(float(p) * (0.44 if kk == 0 else 0.48))))
            irgba = Image.merge("RGBA", (r, g, b, a))
        out = Image.alpha_composite(out, irgba)
    return out.convert("RGB")


def _combined_letterboxed_to_wh(use_values, wh):
    W, H = wh
    leg = (
        legend_linear_equation_values(_w_st_comb, _w_el_comb, _b_comb)
        if use_values
        else _leg_comb_sym
    )
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(
        ax,
        _w_st_comb,
        _w_el_comb,
        _b_comb,
        leg,
        show_contour=True,
        highlight_eps=True,
        legend_tex=False,
    )
    im = fig_to_image(fig)
    iw, ih = im.size
    scale = min(float(W) / float(iw), float(H) / float(ih))
    nw = max(1, int(round(iw * scale)))
    nh = max(1, int(round(ih * scale)))
    im2 = im.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (W, H), (255, 255, 255))
    canvas.paste(im2, ((W - nw) // 2, (H - nh) // 2))
    return canvas


def _frames_ch2_24_eps_grid_trim_grow_reveal_combined(*, use_values):
    hold_intro = max(5, _smooth_n(4))
    hold_trim = max(8, _smooth_n(6))
    hold_overlap = max(8, _smooth_n(7))
    hold_end = max(10, _smooth_n(8))

    out = []

    im_full = _epsilon_grid_figure_partial(6, use_values=use_values)
    W, H = im_full.size
    for _ in range(hold_intro):
        out.append(im_full.copy())

    im_trim = _epsilon_trimmed_grid_figure(_best_s123, use_values)
    for _ in range(hold_trim):
        out.append(im_trim.copy())

    for t in np.linspace(0.0, 1.0, _smooth_n(28)):
        out.append(_grow_overlap_frame(float(t), _best_s123, use_values))

    im_stack = _overlap_three_full_bleed(_best_s123, use_values, (W, H))
    for _ in range(hold_overlap):
        out.append(im_stack.copy())

    im_combined = _combined_letterboxed_to_wh(use_values, (W, H))
    last = out[-1].convert("RGB")
    for u in np.linspace(0.0, 1.0, _smooth_n(20)):
        out.append(Image.blend(last, im_combined, float(u)))

    for _ in range(hold_end):
        out.append(im_combined.copy())

    return out


def _epsilon_grid_figure_partial(n_filled, *, use_values, show_indices=None):
    """First ``n_filled`` panels (build order 1…6), **16∶9** canvas (same as ``ch2_23_epsilon_grid_2x3_build_seq_16x9``).

    If ``show_indices`` is set (subset of ``{0,…,5}``), draw **only** those panels — same canvas, unused slots stay blank.
    """
    FW, FH, _ = _epsilon_169_cell_geometry()
    cell_w, cell_h = float(EXPORT_FIGSIZE[0]), float(EXPORT_FIGSIZE[1])
    gap_x, gap_y = 0.05, 0.045
    content_w = 3.0 * cell_w + 2.0 * gap_x
    content_h = 2.0 * cell_h + gap_y
    left0 = 0.5 * (FW - content_w)
    bottom0 = 0.5 * (FH - content_h)
    fig = plt.figure(figsize=(FW, FH))
    order = [
        (0, 0, epsilon_variants[0]),
        (1, 0, epsilon_variants[1]),
        (0, 1, epsilon_variants[2]),
        (1, 1, epsilon_variants[3]),
        (0, 2, epsilon_variants[4]),
        (1, 2, epsilon_variants[5]),
    ]
    for idx, (r, c, spec) in enumerate(order):
        if show_indices is not None:
            if idx not in show_indices:
                continue
        elif idx >= int(n_filled):
            break
        left_m = left0 + float(c) * (cell_w + gap_x)
        bottom_m = bottom0 + float(1 - r) * (cell_h + gap_y)
        ax = fig.add_axes([left_m / FW, bottom_m / FH, cell_w / FW, cell_h / FH])
        _, ws, we, bb, leg_v = spec
        if use_values:
            leg = legend_linear_equation_values(ws, we, bb)
        else:
            leg = leg_v
        draw_probability_panel(
            ax,
            ws,
            we,
            bb,
            leg,
            show_contour=True,
            highlight_eps=True,
            legend_tex=False,
        )
    return fig_to_image(fig)


def _epsilon_trimmed_grid_figure(s123, use_values):
    """Three surviving ε panels on the **16∶9** grid (unused slots blank)."""
    return _epsilon_grid_figure_partial(
        6, use_values=use_values, show_indices=set(_epsilon_keep_indices(*s123))
    )


save_gif(
    _frames_ch2_24_eps_grid_trim_grow_reveal_combined(use_values=True),
    "ch2_24_epsilon_grid_trim_grow_reveal_combined_values.gif",
    duration=_gif_dur(38),
)
save_gif(
    _frames_ch2_24_eps_grid_trim_grow_reveal_combined(use_values=False),
    "ch2_24_epsilon_grid_trim_grow_reveal_combined.gif",
    duration=_gif_dur(38),
)


## Late story — mistakes, sensitivity, calibration — **`ch2_25`–`ch2_34`**

Step-by-step assets for **`script_ch2.md`** from “making mistakes” through “continuous feedback”:

**Sensitivity / fixing mistakes**
- **`ch2_25_fix_via_b_vs_w1.gif`**: bad threshold (3 mistakes) → fix via **`b`** vs **`w₁`**.
- **`ch2_28_threshold_slips_bad.gif`**: slide **`b`** until mistakes **appear** (threshold “slips”).
- **`ch2_29_epsilon_w1_pingpong.gif`**: **`w₁ ± ε`** → baseline → **`w₁ − ε`** (script ±ε on **`w₁`**).
- **`ch2_30_epsilon_w2_pingpong.gif`**: same pattern for **`w₂`**.
- **`ch2_31_epsilon_b_pingpong.gif`**: same pattern for **`b`**.

**Severity (one mistake, different σ)**
- **`ch2_26_one_mistake_uncertain_vs_confident.png`**: side-by-side still.
- **`ch2_32_severity_three_steps.gif`**: full-frame **uncertain** → **confident** → **side-by-side** recap.

**Continuous feedback / flat mistake count**
- **`ch2_27_micro_nudge_w1_probabilities.gif`**: sweep **`w₁`**, contours move, **0** mistakes.
- **`ch2_33_micro_cycle_w1_w2_b.gif`**: tiny nudges **`w₁` → `w₂` → `b`** through baseline.
- **`ch2_34_micro_b_oscillation_flat_mistakes.gif`**: oscillate **`b`** — mistake count **flat**, probabilities **change**.

On-frame text uses **`NOTE_SIZE`** (axis-area notes, not plot titles).

In [8]:
# --- Late-story helpers/config (assets are saved in the following single-purpose cells) ---
_BASE_BAD = (1.0, -1.0, -1.02)
_FIX_B = (1.0, -1.0, -0.97)
_FIX_W1 = (1.08, -1.0, -1.02)
_PAIR_UNCERTAIN = (0.8, -1.55, 3.05)
_PAIR_CONFIDENT = (1.2875 * 3, -1.835 *3, 4.0*3)


def _late_story_frame(ws, we, bb, note, *, highlight=True):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_probability_panel(
        ax,
        ws,
        we,
        bb,
        legend_linear_equation_values(ws, we, bb),
        show_contour=True,
        highlight_eps=highlight,
    )
    ax.text(0.02, 0.98, note, transform=ax.transAxes, va="top", fontsize=NOTE_SIZE)
    return fig_to_image(fig)


def _mistake_n(ws, we, bb):
    return int(np.sum(mistake_mask(study_sep, exam_sep, y_sep, float(ws), float(we), float(bb))))


def _frames_ch2_25():
    frames = []
    for ws, we, bb, note in (
        (*_BASE_BAD, r"baseline (3 mistakes)"),
        (*_FIX_B, r"nudge $b$"),
        (*_BASE_BAD, r"baseline again"),
        (*_FIX_W1, r"nudge $w_1$"),
    ):
        frames.append(_late_story_frame(ws, we, bb, note, highlight=True))
    return frames


def _fig_ch2_26():
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(EXPORT_FIGSIZE[0] * 1.62, EXPORT_FIGSIZE[1]))
    draw_probability_panel(axL, *_PAIR_UNCERTAIN, legend_linear_equation_values(*_PAIR_UNCERTAIN), show_contour=True, highlight_eps=True)
    axL.text(0.02, 0.98, r"mistake near $\sigma \approx \frac{1}{2}$", transform=axL.transAxes, va="top", fontsize=NOTE_SIZE)
    draw_probability_panel(axR, *_PAIR_CONFIDENT, legend_linear_equation_values(*_PAIR_CONFIDENT), show_contour=True, highlight_eps=True)
    axR.text(0.02, 0.98, r"mistake with higher confidence", transform=axR.transAxes, va="top", fontsize=NOTE_SIZE)
    fig.subplots_adjust(wspace=0.18)
    return fig


def _frames_ch2_27():
    out = []
    for w_s in np.linspace(0.92, 1.08, _smooth_n(32)):
        out.append(_late_story_frame(float(w_s), W_EL0, B0, "", highlight=False))
    return out


_bs_slip = np.concatenate(
    [
        np.linspace(-0.55, -0.995, _smooth_n(9)),
        np.linspace(-0.9988, -1.00055, _smooth_n(14)),
        [-1.02],
    ]
)


def _frames_ch2_28():
    out = []
    for bb in _bs_slip:
        bb = float(bb)
        out.append(_late_story_frame(1.0, -1.0, bb, rf"$b={bb:.4f}$ — mistakes: {_mistake_n(1.0, -1.0, bb)}", highlight=True))
    return out


_PING_STEPS = max(10, _smooth_n(8))


def _frames_ch2_29():
    seq = np.concatenate([
        np.linspace(W_ST0, W_ST0 + EPS, _PING_STEPS, endpoint=False),
        np.linspace(W_ST0 + EPS, W_ST0, _PING_STEPS, endpoint=False),
        np.linspace(W_ST0, W_ST0 - EPS, _PING_STEPS, endpoint=True),
    ])
    return [_late_story_frame(float(ws), W_EL0, B0, "", highlight=True) for ws in seq]


def _frames_ch2_30():
    seq = np.concatenate([
        np.linspace(W_EL0, W_EL0 + EPS, _PING_STEPS, endpoint=False),
        np.linspace(W_EL0 + EPS, W_EL0, _PING_STEPS, endpoint=False),
        np.linspace(W_EL0, W_EL0 - EPS, _PING_STEPS, endpoint=True),
    ])
    return [_late_story_frame(W_ST0, float(we), B0, "", highlight=True) for we in seq]


def _frames_ch2_31():
    seq = np.concatenate([
        np.linspace(B0, B0 + EPS, _PING_STEPS, endpoint=False),
        np.linspace(B0 + EPS, B0, _PING_STEPS, endpoint=False),
        np.linspace(B0, B0 - EPS, _PING_STEPS, endpoint=True),
    ])
    return [_late_story_frame(W_ST0, W_EL0, float(bb), "", highlight=True) for bb in seq]


def _frames_ch2_32():
    frames = [
        _late_story_frame(*_PAIR_UNCERTAIN, r"one mistake — $\sigma \approx \frac{1}{2}$ at error", highlight=True),
        _late_story_frame(*_PAIR_CONFIDENT, r"one mistake — higher confidence at error", highlight=True),
    ]
    fig = _fig_ch2_26()
    frames.append(fig_to_image(fig))
    return frames


_MICRO = 0.14
_cycle = [
    (W_ST0, W_EL0, B0, "baseline"),
    (W_ST0 + _MICRO, W_EL0, B0, r"small $\Delta w_1$"),
    (W_ST0, W_EL0, B0, "baseline"),
    (W_ST0, W_EL0 + _MICRO, B0, r"small $\Delta w_2$"),
    (W_ST0, W_EL0, B0, "baseline"),
    (W_ST0, W_EL0, B0 + _MICRO, r"small $\Delta b$"),
]


def _frames_ch2_33():
    return [_late_story_frame(ws, we, bb, rf"{note} — mistakes: {_mistake_n(ws, we, bb)}", highlight=False) for ws, we, bb, note in _cycle]


def _frames_ch2_34():
    out = []
    for bb in np.linspace(-0.45, -0.35, _smooth_n(28)):
        bb = float(bb)
        out.append(_late_story_frame(1.0, -1.0, bb, rf"$b={bb:.3f}$ — mistakes: {_mistake_n(1.0, -1.0, bb)} (count unchanged)", highlight=False))
    return out


In [11]:
save_gif(_frames_ch2_25(), "ch2_25_fix_via_b_vs_w1.gif", duration=420)

In [9]:
save_fig(_fig_ch2_26(), "ch2_26_one_mistake_uncertain_vs_confident.png")

In [13]:
save_gif(_frames_ch2_27(), "ch2_27_micro_nudge_w1_probabilities.gif", duration=_gif_dur(38))

In [14]:
save_gif(_frames_ch2_28(), "ch2_28_threshold_slips_bad.gif", duration=_gif_dur(300))

In [15]:
save_gif(_frames_ch2_29(), "ch2_29_epsilon_w1_pingpong.gif", duration=_gif_dur(380))

In [16]:
save_gif(_frames_ch2_30(), "ch2_30_epsilon_w2_pingpong.gif", duration=_gif_dur(380))

In [17]:
save_gif(_frames_ch2_31(), "ch2_31_epsilon_b_pingpong.gif", duration=_gif_dur(380))

In [18]:
save_gif(_frames_ch2_32(), "ch2_32_severity_three_steps.gif", duration=_gif_dur(520))

In [19]:
save_gif(_frames_ch2_33(), "ch2_33_micro_cycle_w1_w2_b.gif", duration=_gif_dur(360))

In [20]:
save_gif(_frames_ch2_34(), "ch2_34_micro_b_oscillation_flat_mistakes.gif", duration=_gif_dur(42))

## Keras logistic gradient descent (Chapter 1 export 79 analogue) — **`ch2_35`** / **`ch2_36`**

**Prerequisites:** Cell 1 (setup: **`ST`** / **`EL`**, **`ST_3D`** / **`EL_3D`**, **`CMAP_GD`**, **`draw_dataset`**, **`boundary_line_xy`**, **`SIG3D_ADJ`**, **`EXPORT_FIGSIZE`**, …).

- **`ch2_35_keras_logistic_gradient_descent_plane.gif`** — Same training recipe as **`logistic-regression-chap1.ipynb`** Scene 8d **`79_logistic_gradient_descent_plane.gif`**: **`tensorflow.keras`** **`Sequential`**, **`Dense(1, input_dim=2, activation="sigmoid")`**, **`binary_crossentropy`**, **Adam** (`learning_rate=1e-1`); **`study_sep`**, **`exam_sep`**, **`y_sep`**; fit in **1-epoch chunks** for **40** epochs; **`predict`** on the fine **`ST`/`EL`** mesh for **`contourf`** + **`draw_dataset`**; **duplicate frames** + **tail hold** like export 79.
- **`ch2_36_keras_logistic_gradient_descent_sigmoid_3d.gif`** — **Same model each frame**, **3D** **`plot_surface`** of σ over **`ST_3D`/`EL_3D`** (camera **elev 32°**, **azim −152°**); scatters at each student’s σ; **z = 0.5** decision segment in the box when defined.

**Workflow:** Run the **training** cell (re-seeds and retrains), then each **save** cell exports one GIF.

In [42]:
# Keras single-layer logistic (2 → 1, sigmoid) + binary CE — train once, fill frame lists for ch2_35 / ch2_36.
# Same loop as chap1 Scene 8d export 79: 1-epoch chunks, snap after each chunk; separable roster only.

import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

GD_MS = 40
GD_HOLD_LAST = 18

_xa, _xb = float(xlim[0]), float(xlim[1])
_ya, _yb = float(ylim[0]), float(ylim[1])

mesh_plane = np.c_[ST.ravel(), EL.ravel()].astype(np.float32)
shape_plane = ST.shape
mesh_surf = np.c_[ST_3D.ravel(), EL_3D.ravel()].astype(np.float32)
shape_surf = ST_3D.shape

X_k = np.column_stack([study_sep.astype(np.float32), exam_sep.astype(np.float32)])
Y_k = y_sep.reshape(-1, 1).astype(np.float32)


def _keras_wb(model):
    W, b = model.layers[0].get_weights()
    return float(W[0, 0]), float(W[1, 0]), float(b[0])


def ch2_keras_gd_pack_frames(frames):
    out = []
    for im in frames:
        out.append(im.copy())
        out.append(im.copy())
    last = frames[-1].copy()
    for _ in range(GD_HOLD_LAST):
        out.append(last.copy())
    return out


def _snap_keras_plane(model):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    Z = model.predict(mesh_plane, verbose=0).reshape(shape_plane)
    ax.contourf(
        ST,
        EL,
        Z,
        levels=np.linspace(0.0, 1.0, 45),
        cmap=CMAP_GD,
        vmin=0,
        vmax=1,
        antialiased=True,
        alpha=0.32,
        zorder=1,
    )
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    ax.set_xlim(_xa, _xb)
    ax.set_ylim(_ya, _yb)
    ax.grid(alpha=0.2)
    return fig_to_image(fig, tight_layout=False)


def _snap_keras_sigmoid_3d(model):
    w_st, w_el, bb = _keras_wb(model)
    Zs = model.predict(mesh_surf, verbose=0).reshape(shape_surf)
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    fig.subplots_adjust(**SIG3D_ADJ)
    ax.plot_surface(
        ST_3D,
        EL_3D,
        Zs,
        cmap=CMAP_GD,
        vmin=0,
        vmax=1,
        alpha=0.42,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    zs_pts = model.predict(X_k, verbose=0).ravel()
    for i in range(n_sep):
        c = PASS_COLOR if y_sep[i] else FAIL_COLOR
        ax.scatter([study_sep[i]], [exam_sep[i]], [zs_pts[i]], color=c, s=55, depthshade=True, zorder=6)
    bxy = boundary_line_xy(w_st, w_el, bb, _xa, _xb, _ya, _yb)
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, np.full_like(bx, 0.5), "k--", linewidth=1.4, zorder=4)
    ax.view_init(elev=32.0, azim=-152.0)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=11)
    ax.set_zlabel(r"$\sigma(\mathrm{logit})$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)
    return fig_to_image(fig)


model_keras_gd = keras.models.Sequential()
model_keras_gd.add(keras.layers.Dense(1, input_dim=2, activation="sigmoid"))
model_keras_gd.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-1),
)

CH2_KERAS_GD_SNAPS_2D = []
CH2_KERAS_GD_SNAPS_3D = []
_epochs, _sf = 100, 1
for _ in range(_epochs // _sf):
    model_keras_gd.fit(X_k, Y_k, epochs=_sf, verbose=0, batch_size=100)
    CH2_KERAS_GD_SNAPS_2D.append(_snap_keras_plane(model_keras_gd))
    CH2_KERAS_GD_SNAPS_3D.append(_snap_keras_sigmoid_3d(model_keras_gd))

CH2_KERAS_GD_MS = GD_MS

/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [43]:
save_gif(
    ch2_keras_gd_pack_frames(CH2_KERAS_GD_SNAPS_2D),
    "ch2_35_keras_logistic_gradient_descent_plane.gif",
    duration=CH2_KERAS_GD_MS,
)

In [44]:
save_gif(
    ch2_keras_gd_pack_frames(CH2_KERAS_GD_SNAPS_3D),
    "ch2_36_keras_logistic_gradient_descent_sigmoid_3d.gif",
    duration=CH2_KERAS_GD_MS,
)